<!-- dd:dd-lesson-ar-02 -->

# ARENA 0.2 — Neural networks and ResNets

*PyTorch · `ar-02`*

Work through this with the **Delta Drills** side panel open. It picks what you practise, sends you to the cell, and records how it went — you do not need to read this notebook in order.


In [ ]:
# === Delta Drills ===
# Which lesson this notebook is, for the side panel. Nothing to run.
DD_LESSON_ID = "ar-02"


In [ ]:
#@title 🔧 Delta Drills checker — run me first { display-mode: "form" }
# Delta Drills — problem checker. Generated; see scripts/colab_grader.py.
#
# `dd_check(<problem id>)` runs your `solve` against the same cases the tutor
# grades with, and tells you which ones failed. It reads `solve` out of the
# notebook, so define it (run your cell) before you check.
import base64
import json
import sys
import zlib

import numpy as np

# Filled in by the generated cell that follows this source: {qid: {fn, cases}}.
_DD_TESTS = {}

# Where the ARENA digits fixture is fetched from, also filled in by that cell.
_DD_FIXTURE_URL = ""
_DD_FIXTURE_PATH = "/delta_numbers.npy"

_DD_RTOL = 1e-5
_DD_ATOL = 1e-6


def _dd_install_fixtures():
    """Make `np.load('/delta_numbers.npy')` work here the way it does in the app.

    24 of the einops drills are written against the ARENA digits image, and the
    bank refers to it by an absolute path the backend rewrites at grade time
    (`code_runner.CODE_PREAMBLE`). Nothing rewrote it in a notebook, so those
    problems could not run at all in Colab — not the checker, not the starter
    code the learner was sent there to fill in. Downloaded on first use, so the
    six notebooks that never touch it never pay for it.
    """
    import os
    import urllib.request

    original = np.load
    if getattr(original, "_dd_patched", False):
        return

    def _load(file, *args, **kwargs):
        if str(file) == _DD_FIXTURE_PATH and not os.path.exists(_DD_FIXTURE_PATH):
            if not _DD_FIXTURE_URL:
                raise FileNotFoundError(
                    "This drill needs the ARENA digits fixture and no source was "
                    "compiled into this notebook — regenerate it."
                )
            urllib.request.urlretrieve(_DD_FIXTURE_URL, _DD_FIXTURE_PATH)
        return original(file, *args, **kwargs)

    _load._dd_patched = True
    np.load = _load


def _dd_load(blob):
    """The test payload, deflated and base64'd.

    Not encryption and not pretending to be — it is one `zlib.decompress` away.
    It is compressed because the payload for a 84-problem notebook is ~80 KB of
    JSON, and out of sight because an expanded grader cell would otherwise sit
    in the notebook spelling out the expected answer to every problem below it.
    """
    return json.loads(zlib.decompress(base64.b64decode(blob)).decode("utf-8"))


def _dd_preflight_torch():
    """Import torch once, here, where a failure can still be explained.

    Every drill cell opens with `import torch as t`, so the learner meets a
    broken torch install as a traceback through torch's own internals — the one
    reported was `AttributeError: partially initialized module 'torch' has no
    attribute 'fx'` from `torch/_export/utils.py`, raised while evaluating a
    function's annotations. That message names neither the cause nor the cure,
    and it is not even the real error: it is what a LATER import sees after an
    earlier one died partway and left the half-built module in `sys.modules`.
    Python does unwind a failed import normally, but a torch that was swapped
    on disk under a running kernel (a `pip install` mid-session) or shadowed by
    a stray `torch.py` gets far enough in to be cached before it falls over.

    So: purge the wreckage and retry ONCE, which is the whole fix whenever the
    first failure was transient, and report what actually broke when it is not.
    Importing torch in this cell rather than lazily is safe now in a way the
    `_dd_tensor` comment below still guards against for the per-comparison
    path — the bank is 448/448 torch and every notebook imports it a few cells
    down, so there is no numpy-only notebook left to charge for it.

    Never raises: a checker that refuses to load over this would take the
    lesson down with the runtime.
    """

    def _purge():
        # Submodules too, and that is the whole point. Python drops only the
        # module that raised, so `torch` goes and a `torch._export` imported
        # seconds earlier STAYS — and the next `import torch` re-runs
        # `torch/__init__.py` straight back into that stale submodule, which
        # reaches for a `torch.fx` the half-built parent has not bound yet.
        # Leaving one behind reproduces the bug instead of clearing it.
        for name in [n for n in sys.modules if n == "torch" or n.startswith("torch.")]:
            del sys.modules[name]

    def _usable(mod):
        # `import torch` does NOT re-execute a module already in sys.modules,
        # so a corpse left by a failed import is imported "successfully" and
        # the error surfaces later, from the learner's own cell. Judge the
        # object, not the statement: a torch that finished has both of these.
        return hasattr(mod, "fx") and hasattr(mod, "__version__")

    cached = sys.modules.get("torch")
    if cached is not None and not _usable(cached):
        _purge()

    for attempt in (1, 2):
        try:
            import torch
            if not _usable(torch):
                raise ImportError(
                    "torch imported but is only partially initialised "
                    "(no .fx) — an earlier import in this session died partway"
                )
            return True
        except Exception as exc:
            if attempt == 1:
                _purge()
                continue
            print(
                "⚠️  This runtime cannot import PyTorch, so no drill in this "
                "notebook will run.\n"
                "    %s: %s\n"
                "    Fix: Runtime ▸ Disconnect and delete runtime, then reopen "
                "this notebook and run\n"
                "    this cell first. If it comes back, check for a file named "
                "torch.py in /content,\n"
                "    and re-run any pip install BEFORE anything imports torch."
                % (type(exc).__name__, exc)
            )
    return False


def _dd_tensor(value):
    # torch only if something already imported it. numpy-only notebooks must
    # not pay a torch import to compare two lists of ints.
    torch = sys.modules.get("torch")
    return torch is not None and isinstance(value, torch.Tensor)


def _dd_close(a, b):
    """Tolerance compare, but ONLY when a float or complex is involved.

    torch defaults to float32 where numpy defaults to float64 and honest
    answers differ in reduction order, so exact equality fails correct work.
    Integer and boolean results stay exact — an index answer (argmax, nonzero,
    searchsorted) must never be fudged by a tolerance. Returns None to mean
    "not a float comparison, use exact equality".
    """
    try:
        floaty = any(
            np.issubdtype(x.dtype, np.floating) or np.issubdtype(x.dtype, np.complexfloating)
            for x in (a, b)
        )
        if not floaty:
            return None
        if a.shape != b.shape:
            return False
        return bool(np.allclose(a, b, rtol=_DD_RTOL, atol=_DD_ATOL, equal_nan=True))
    except Exception:
        return None


def _dd_array_equal(a, b):
    close = _dd_close(a, b)
    if close is not None:
        return close
    return bool(np.array_equal(a, b))


def _dd_equal(a, b):
    if _dd_tensor(a) or _dd_tensor(b):
        try:
            a2 = a.detach().cpu().numpy() if _dd_tensor(a) else np.asarray(a)
            b2 = b.detach().cpu().numpy() if _dd_tensor(b) else np.asarray(b)
            return _dd_array_equal(a2, b2)
        except Exception:
            # dtypes numpy cannot hold (bfloat16, conj views): equal tensors
            # must not grade as unequal — ask torch itself.
            torch = sys.modules.get("torch")
            if torch is not None and isinstance(a, torch.Tensor) and isinstance(b, torch.Tensor):
                try:
                    return bool(torch.equal(a.detach().cpu().resolve_conj(),
                                            b.detach().cpu().resolve_conj()))
                except Exception:
                    return False
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        return _dd_array_equal(np.asarray(a), np.asarray(b))
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        if len(a) != len(b):
            return False
        return all(_dd_equal(x, y) for x, y in zip(a, b))
    close = _dd_close(np.asarray(a), np.asarray(b))
    if close is not None:
        return close
    return bool(a == b)


def _dd_seed():
    # The same seed before the actual-side and the expected-side setup runs, for
    # BOTH rngs: setup executes twice, so an unseeded torch.rand in a fixture
    # would hand the two sides different data and fail an honest answer.
    np.random.seed(0)
    torch = sys.modules.get("torch")
    if torch is not None:
        torch.manual_seed(0)


def _dd_show(value, limit=320):
    try:
        text = repr(value)
    except Exception as exc:
        text = "<unrepresentable: %s>" % type(exc).__name__
    text = " ".join(text.split()) if len(text) > limit else text
    if len(text) > limit:
        text = text[: limit - 1] + "…"
    return text


def dd_check(question_id, verbose=True):
    """Grade the `solve` you just defined against this problem's cases.

    Returns True when every case passes. Prints which ones did not, with the
    fixture, what was expected and what came back — a failing grade should be
    evidence you can act on, not a verdict.
    """
    qid = str(question_id)
    entry = _DD_TESTS.get(qid)
    if entry is None:
        print("No checker for problem %s in this notebook." % qid)
        return False

    # The learner's namespace, not this function's: `solve` lives in the cell
    # they ran, and in Colab that is the caller's globals.
    try:
        env = sys._getframe(1).f_globals
    except Exception:
        env = globals()

    fn_name = entry.get("fn") or "solve"
    if fn_name not in env:
        print("❌ `%s` is not defined yet — run your solution cell first." % fn_name)
        return False

    cases = entry.get("cases") or []
    failures = []
    for i, case in enumerate(cases, 1):
        # A fresh copy per case: fixtures are exec'd, and exec'ing them into the
        # notebook's own globals would quietly overwrite whatever the learner
        # named `x` two cells ago.
        ns = dict(env)
        try:
            if case.get("setup_code"):
                _dd_seed()
                exec(case["setup_code"], ns)
            actual = eval(case["call"], ns)
            expected_setup = case.get("expected_setup_code") or case.get("setup_code")
            if expected_setup:
                _dd_seed()
                exec(expected_setup, ns)
            expected = eval(case["expected_expr"], ns)
            if not _dd_equal(actual, expected):
                failures.append((i, case, _dd_show(expected), _dd_show(actual), ""))
        except Exception as exc:
            failures.append((i, case, "", "", "%s: %s" % (type(exc).__name__, exc)))

    total = len(cases)
    if not failures:
        print("✅ Problem %s — %d/%d cases passed." % (qid, total, total))
        return True

    print("❌ Problem %s — %d of %d cases failed." % (qid, len(failures), total))
    if verbose:
        for i, case, expected, actual, error in failures:
            print("\n  case %d" % i)
            if case.get("setup_code"):
                for line in case["setup_code"].strip().splitlines():
                    print("    given     %s" % line)
            print("    called    %s" % _dd_show_source(case.get("call", "")))
            if error:
                print("    raised    %s" % error)
            else:
                print("    expected  %s" % expected)
                print("    you got   %s" % actual)
    return False


def _dd_show_source(text, limit=160):
    text = " ".join(str(text).split())
    return text if len(text) <= limit else text[: limit - 1] + "…"

_DD_FIXTURE_URL = "https://raw.githubusercontent.com/AkiraTheSquid/arena-book-colab/main/ARENA_5.0/ch-1-foundations/numbers.npy"
_dd_preflight_torch()
_dd_install_fixtures()
_DD_TESTS = _dd_load(
    "eNrtXVtz2ziy/iuuvMSpkljoxn1Sft3H87RvXteUk2gyqYmdrO0kc87W/vfTDYAkKFF3WZQoKGXHoiiSaPTl6wsa/3kDCPbN"
    "b1f/efPHI/335vnb15+zN5OrNx/vn2fPdOT2P2+eZy8/vv/+8dunGZ/x5eH7t6eXq5dvTx//vLp/vnr512N+qHp8rP748fjx"
    "5cu3x/uvfMI/mhNmXx6/fX/+1+PfNy/V/dP94+fZNVQTkNW76mn2/Of999m1nKh38fZfvzbPc/13ODb7+/vs48vs0+/0xxN/"
    "+FK9zB6fvz1d395CJSZXmn/5StxNrm6R/zb8C0Q8IvmNDUcgHlH8xoUjdPodHfr08r/fZ/R0f3z9dv8i8d2b/06ujksANZHv"
    "qn/uTIIwbJkNTyc68AHbjDcRiUkTyTEkCaTIxq8n5t0t/KYm8Ju+24UMcYbDZEMaZpziQA5MzODDoUAchA4xAs/gUmLcETlY"
    "aNz5Cw20IhJJcMbcPy//Z87FNuPYZXNTc6I/MU58f/XzJp+YyZR+sJqI6q6PSyc/N9Ns01a/JRne4vAJMXeXPIEugUKwM3nq"
    "UdajhswCbnb4xASmS6KGfSZyZxJ12aGxkFscXiWCKMYugro1qnhatuLg4qQbmHh6luTggiEDDjJhYp1dbWgQzh/yYOsD4LlD"
    "njnv5ewxj8sgOa5hRTw2K26jT3cGNI1Bpg/D8anIbHY8pLLzRX1OslPHRTrb6N31JFEtLMFEl2aI3XEGSmF7qCHm8OhmG+28"
    "KZdkI53KfPQtxZJ73+WEllay8fNEbdZWCpccSzwokau26K3fDyOJ9tiM6V0W5fIjCuXUoSpZm4fwzjdQdCUrqxGw8u2C94bZ"
    "VOexPhXJcRs5PpHKzAcAbRYUc/U3/GJUsHanIQ+ZAraxwjOXoA5dG/rNUSmjRaBTTv+cyi4TwYxi4Tv5POWz0aH50HQ9rOT2"
    "c9r8mOdom0Q8USpGvyDNScSFjfibOlSbvunzcC+KPNKLkAd5Edvw7krNoc9ec0Cg+/lKaJzy85YEhf1jqLnMjMA+ySBuNqWS"
    "ktSltzaGsG2jGUXzntStMiPAYSbqIueimqmtZv0+EkA1ykg075kOYhxADX20n95FVkgGMr13KiWZfG1gBbYHWI87sw7IjSFH"
    "LZscle0AhSXIln7L85cPnWWYbYavuhna+XTsCGQiIRbbQSbhHYbRyoRIYghWBQqptZkGNwbfXKQIbNQCMr7TWCPheDiqSNAx"
    "LGhN7QbazuEoLaLWuhqbs9NhBntqBGYmaEoZhqdj0Kd2+GNMu0NSkyQtnlkfjdoWTSKlcc256SjdwOhxSJ+OnGEiCDY6DNL4"
    "eujpsBGNvaL/VDJP6ezmcHznE3+Gi6Sz02E2cWKd5I49TW+kPiV/49B5QdAn544cOhsIUmi/yl8BMwgTNyPsZuR7xvhuvWYQ"
    "c8n+I05ot2AjZCx2HMTtQIUtzUyoZgT7zcIKjQn26FUVv9ohEoFJQ27Hfb944NWn2cv9xz+v11FgOp++OdI0ZmPk+Zsb5C1m"
    "cdA257T3aOuEncozVcf1brKBi0rPjdsdYEbVPtPYYb3Ok3z98vxy/enLx5fr+pmqx/sHeoLvZCAeZi+zp+frd+/6H+727a/Z"
    "l89/vry92/sxmpvHK5JF+vePL2SWfv/8dP+p9+b/fPoxa4UZBxVmmu4PHcafm3+GP/1m89fkwzZMEHO2djiJDiP9n2+PswXB"
    "ZlxwgBHWojyY9AbhzWZTzk8mQSB1iLmUbX5nuZGCwY1URgrYxWptR5Sp6dSiDGm4OjzQb8mCwTkE0zfp9gENVq7BFvGJO8j0"
    "rmN3eQoOgD6AA6Dr1O95OgAwrAMgKkwE3HUS8NgT8HCTnq4DrmaP1wFgPVQbwCnRyoEqvknxTS7EN9mbVWvR+9UZxu1S4Ztc"
    "tU7PQ/J2Pvz444/08V2/r0Ozdvv2ma4+e3t310qqHrOkiqaEWF+AoMLCGp2pGpecukqfnJwuyuJGkYcPX+6f32ZyaM4+dDzF"
    "U4kdbyRbG8eRF692fFBGzHf9Uv3f7Onb8zW+67yV3bfwrmtFHir+87n68jJ7uGbTcVirImnG2bCEmwTDcjx02h3aOkhqR2/o"
    "LgaRdpZvFDx6pFh5wI5VGzGvBcsNKFhzS8XuXlXK1BEz3L860QTdO7rXkq+Q5J4OpUhuZc9Ib5uVGDEstL8OQdUu/xpGddzm"
    "XHtQbLkoo77EY0o8psRj9ovHrBHGydVDb/Z3CXJuxXRy9Y/7r8+zVlyduAysetyF+UOJq8iXqothk8OvI65YSjt6Sjt2uO37"
    "VvW8Z5c+q7/g8ot31fMP8nSrD/cf//p1//Sp6w4Hul83Ooifrp82vDCw1TVQoEGBBgUabMKqvVph9vfL0/3vT7PvT0uePWmi"
    "G7rC2yOwG5N9vwflS7xtFYQeMjLdOklNUiOKyvurXzd9Fb95h7tGvOqvfFiIHNQlE/0l4VwdstGytbqJiM5yLyau5dYD1SBg"
    "GxlJHvWvm952qKnQoksd7BfVjUgyHa5ypMMuMisCSUFDmOtgd3e3lDBtJ5k++qyyUJvxTVz6J7NHrWFh1z7IprRqVeWRM2OR"
    "0l56bpapmULe7qexhUM0fNlVCnca/VSOUeB2Y4QoVnlP66npiBVkH6m1YmWL8Vtr/ERm+0TWR+vyLJ8ohm8Xwyf2sHuuCOjK"
    "yAwYflllQHqNRvH1nAwvkMpoIqG6MDGN9yxSulHDyeZJ9eoV3OB8kcS1RQvRT8S88+TU1Obz4vxEWQXlZLxTWjsQWhfrudZ6"
    "hmoA1UQadJgYXT9s+1n4yPInq6TWiyK1q6U2GkvU0hmhrDR0WZ2sasu3TPVoVAVopUgZqnqCer5v0kHlEa0XWnL/vzlRKNi5"
    "SP+yppxT6I0adfaxaNtEr5Z/KPK/Cj/T02Xk+/L4YtSoxVAMMNhzEkB+tsCFdIO7Hs6oxQqLWJWkyVYkmcJgQepzEsD+rEnd"
    "+nshdrTK9Mkio2tbdSj2ItxQ0KzkME9aGGVYpo6V1SvlTBU5W6XRTGqB2uZP0F1m7qSzMUsRvdWe4BTy7QDqTWySJ2iyPQWm"
    "bR/lNb5gqfRZRXKTunKgpf8vTDpLxmQDnyZ2a3f5LhvarQOhpsjc2j2OEHIfUcIFWse5OxdhXJ8myXYNMD3W0dcPi82ZqyW1"
    "lAJdRAXeQfObRVKPW7CHYth8xiLdVYfunX04l65k3rC/HSbazy0laT8bMqKYidRuQ8wuMNAQbiFx221cWXO3x2zVUjRghHdC"
    "96fbc/CdG8Duw3nTHg5fLpJ4diJJqhBuetpJ1tKVL+ZsP2v0I8wryGWt4ElDwuQDbLiuS3fM6skIdy+xsLvfcGf75iVEatsh"
    "7E4o0ewhdXqaY55OGZhbpEjDZnsQQ+bs3VaenZIGWuQd3mBlcht3I6E/4pYkPQwzSZfbgz7Q2M/pfEJpbbkjClm0GhxGWPN1"
    "A0WlLVdpomi0eTYt+qxXn4mu47eBNlNFm0GKDuDqsEtzBdyOuSe/cPIB108iZFoRRq4Ve6ktusEY7HZC25/CYrDw8WHV6CpW"
    "zYM87WZYi+SsZ2ONKd+QsrK5ns62jZeDw/FX0tSLE5A2CYyXwfYCuLCV4P7EnnYUhdrIE9dFy5+Elo/7r1a6qPWDqXUxHDVH"
    "rdRlaj5aP0VR4AdS4KZdq7VcYZuisE9CYaey/wGWRoxRVQ9DyVGraRyMQceom6GfmrVWtkUr7xf6FePGv/vRRQ9cA30SoV4Y"
    "dgfCkwvuJmHWy8WmVk6uKKf9lBOMFevtTpHzB2z7cAOMCljtroL618LXescXvVMySCWDVLzY/SjbuV5JIb1uCikvK4xqY10K"
    "CURR86eh5kXlvdVoFQqNwjpem0gXFqgsGpTK0QceXByalSC00/QFo4y3vGtjhcZ5BUoiSmtQQzERBzQRY6k+PUUTAcDsrNAC"
    "emJiExkftCM2ViDQSeGcDY1OjXEgpQGjlFXKJ2EgobEajJHkLQvhojDQlyVoI0lo0LooS8oLICkRAEZI7VUxP4c0Pz3aisk+"
    "r9bipHl6aaUdoBE0qRhyZ9JbZxx9XUmQdjZd1javNl5QjNdpGK8p5ndUJ7VYqVigYoE22WUpl79pW2pbr14sduJgbopZXKNh"
    "1vkpZdHZdqp+1SRt2Nu6s8XhJcec9qXlpertvegm53b6m2sodZkaeS+KQs9SktVKt6yJO5ngULuFe/ZgBV8fMglQ8PUrJgE6"
    "HmImkFNd4PXBwzBdmzmFdYpeDqLo75/uHz/PrpmesnpXPc2e/7z/PrvGCUzMfPeajIY8Wy0V99tXzqZeRsTvSSzCERB1h0X6"
    "3XSvAZs6e8YWZvy77lWsU5vP2M6Mfy/vbPNqfL2EnjBBouftbxP69xve9RA2MOYU9iSoX9N36wjyPA1MMZFRkhZ7IN3iPgOM"
    "cS1M4pW6pqyWK3VxcuWSMOm2aSIev//7sKIAQy5jf1UZWFopUTO8PkGG/34j+kb7fcPdWTptuVI5b2MTbGrMW1sRUdsSbix7"
    "uibg+w3sSJIWHMq54ub4UdeiJoR5QrKwOzM0TfM69SsrpcGcnjRsqv3fP9/A+yXE+jV53oRgF4uw1lkVoi2+XyKCG9L2JMFW"
    "n51JY92HjyLuMmulzV4c2Eq9bmNT5ehg21RmF7dPCZ/oWBZmjw5KhkVhmYiMEYr1eyOr5MMVJ784+cXJ3wDftlIl1lodf3lW"
    "BxqRYbGQrShcko8/WsOyrhRDiTPm+PdXz0txdzEnuwsR0xV3pqs/A3uy3wg39GEUXFzEQI3WX8fzd8Ox5Uy86FhWp/ioG/Fc"
    "qEva5uxLjV/FAtupv4gAltjIk1DygiUsNrzEo3f9OxWJgAG78R1JDpZ1f6nZX12ygUnR2xBzkKLxBrjwMeYfg5+gYgZSnKqv"
    "/fqG4xSDuq9jNTKPYZXU6Es2Gkfu7HMqUjBQO6Mjcf6KTn16YPdYwty0y4neq6ZgWYVJs8llT31J29whpf9iyEnlmpEubNrU"
    "ILgsCJWukIJRWIek6HeEYOnu0RShbYNU6GvjdDec36L8nL5RjejtXdrS46D1LBjYvOyl+Zqsp6aekM6nc7HBFe5i60Rs8LAp"
    "vmjbqcs/TRHHGLtfGI0M35LhW2phPCqyqm4zzGufeLDCevIzo/qFVN+9Xw1Qtu1W3YxJLR9gbaP9KWqtX/MrEDpl7o2tjqQT"
    "iXT7B1Julc9qEnSzqTYrvIh4AkfZKFXytJXNIg3jEgFMFJsEOZ1Abf+xIeO+8ZNOlf+00dJxgYjP6wHSap1UF5IkvjYjvslI"
    "5FpFq3oVdOBwW98mCn57Yd2oknAxEdMj6TGSlrCnJ/9zs9adlL14Oy5jXqkMNBZlUJRBUQZFGbAyEEUZFGXwispA5Lqgm6Pa"
    "WRGIXA10r1l0wOJcrFMB8sJVwDW3/rpIqb2WJGVjYftrbqydsbUqlq0pqIMB8zWna6mgjn6xcYim3w/Zg+GYlmFtvyuti/xk"
    "N1Jh+5C532mjzL5PCjLs9UhE2vwveoY272JDIFGzO9e02Ambk8i8xlvVoqrrdjuxYQlguGztGBqdX9eESZkqV19X2uzCFrIL"
    "I7RXLviwVgSmKIJ0IyuPXSNzBmJtY55P6AE3mDqizKQeWMvFxRZxqW+kfMGdy/oj5Y2otCiwsxYfV8Qnm5pYYDH1dfe9ua61"
    "pk36YwlHLiViVtMydfki9WnsDxRoGrBhwqRTqBPsKl9mJdp0ewSevq5DiAkJUQeJw1udmhRCc/GUCanDkSb3P6exXmY6yESe"
    "MvwstQvtjWKCwsohF26csKinXGMtYFhXGCXxi/H/OnEQs4kRt16IuCULsnYRoClZwQF3PTgPCIveoLDSO7AGvETFWTgrpPVo"
    "JBg0vJdONJoSnCJgZ/hkVMaH7XWMA20VGO0QhNXqMkQQ1sBfA0XyStS1RF0vHvYaPWzn+0AQXm8xVTVHTjX9mIZnJ1MbCdJD"
    "iE12vHHxJbUXxjtUw4guc/GEfrrCKydqt2GZqJ70gP1mUS+ocJVrol3GZepFHKt51o2aZ3EoJHhgHoV2Bc+omBTa7jYrudSc"
    "DZe+/4tMzQoc9ddWEbDp/JYYd6fGzjReuQrwbDxe22zs0GylNs2gi84qD9s+SncnKw2JD3BfPrANyPPrAYi9ODHJV7CNX0j6"
    "GsMUAdlCQPyo0Y4YB9ppGXxUWCdGcs2wO0ZNZAz+ENfZPAy0s1OYtbZf10vYistDcTrfgy29s6PGcZClUJvkbRpzuy4kfdq0"
    "Lhi/nWpSyiDX2ikLo7ZTfhx2CkbrlYumbchKLsVRc6maD3eOgWf1OIOdurEjq/hVjhz9Q/3y0nkt1aicAQ+GX87QwJwDMy6N"
    "2+xyu4p/1YXh56moUI8TKE9TItWdMHQ4DPJtetmv5u1x50472upYLQcPjXYj1r0bpnniq6AGwgx3yxsHWjPufD6MLDk6Mi+M"
    "i/AMorWG0BwgCu5cWnnljJMSQVlNPoldrVXtyD20UTCwrcjDdEI40NrRhOvYyFN5qQntKoGIYNzIilO8UCA0OKMM0hBDqQpx"
    "NQorlNHCWb+Ss/3AS/Y6RMEJ/5O7TH6s0NPH7mDfKZGLRXTYtFuOWYJJN0fQVD7KneUVm2a1MPC607pWkws4VVRDsPOwlm0/"
    "UHOqO0FOfX/1+SafmDB+OvrhJm+9z3NNB2ffn28q0evsfJ58mNDHm9WzWFRcPe6tBFACVCjwUKQHnCQ1oEndCe1j1YeTBo2Q"
    "4Kw2IFRIJIiKLKEwDrwHbS0oR+eKSinUIKQximvVvVThKCB4jVpKS3+gdDLG3lVFF5akYjwItE5yG6EKBQpPzyWddWBV4FF6"
    "KEDnjUWnwcTFe44UtKRHBTDhy5z3osujVg619khf4kPaCG/pDEdPFm4c0x6Yngm9lkJKpSxX3sanp1sJhKgHMQ7TesOl+iC1"
    "TUsHmSba0gC1Jcog3yoSjwv3rfJoXHwiprLSznkkayLrxl1EHoFSeWXpQZnOWNEgrFI0IXRFx/CCnhEcqV8vhEKaJDQRWHnt"
    "SFPTJDmNXmjJEmy99cISHCE75bUN5cBGWkMnkB6nuRTeDVkdvKNe60oGfYUUNCGYeeEIZd+11AQB2VM++ks74r5AkpjAOZoY"
    "ASQlzCQkC9Zpb2iGBc0HcVCcKEIKSkgL1nua1zAlhmeSGNc7xVHs+prEbFJZ6QhfGGIKlkYSRkenKEvcwHqLeDeUm8+fGxZf"
    "Lpw6cCH4cpXenVG5MJVpdqOW21vNmcpyfNJpLXmNjdNBoUkkLGeJUJKX3cTVNAJJMVnvWMN5OhdUqyq61wiaDitjjeLV1ii9"
    "VQa4VD9dxCpba7nVqMmfpC16eugaIxUm5OHbw+zx5cfDTdW7g97Tw6Q+Y928SAZZppKCX0Bai6hr5Plppg6dAsMnC92SSu9J"
    "qnopSb0q5DQFeo4QHRJAtScJVuM5YqJzx3P94vb0MztKBPebYD8i69PPzS0cAR1FSoxAj1Kk1qwxaX8EghiesB5pSU1QgjSi"
    "jgWdnrCJIOVGBgwAA+LynmGQMATNjCGbyHDNe4YbitAOOcvKNL1KyTYSOKP/yIISvov20JIVI0fbkwdqXYAthMZIrUqymmzl"
    "rE3RJAGkmAnaGL66lWEFJOMyMHR/glFkZvkYfdsB30DSyaSwG8QHNCpnyJ+nR/W8Kt6Fng+k8Qkc0sN7QnnSha0eyDBYJxwh"
    "RxFc37hczhOulKgFB38I8wVHkddj0gUISBIAVipsG8vBA0JeBPiEIzNft0Vg6hE5nLAEJIlm0YpIQp9WCkVgF1kPRtNiJF2Q"
    "AC/dkYCiToafaE2PSQRAZ4kW0fJLMmNAcJGIylCEjhgCffSjgS5MI70Q1LdUG3ckKaxNm+AGOHE7WeL8LvtIgqG74LldPJYS"
    "p5ucmVCh1ex6GECC+jzDLB28/Jfwi7NSg3c+SQdJEXkA6HjZGwmPC/4EXdQgyZw3mc8lK8akdAlNPhtJspQssprOQ0W87RjA"
    "2rTqmG5KYFZwSMgFD4mdwWi2CeAIzy7Z2SPNHuvVYZnN8Og27DIF8pNJwbFnSnqIRF3Hes3woqkiN5JcS+YMjsCREiL9FnUp"
    "1Nqsx3Um/BlPJwWoWZfJUOuICrQjSCtBsopfjUmlgBIfmd+ZZ2FfnBK9KNGLEr3YIXpxKttJnUh4oqNUtoo/rCNkrc2xeEc7"
    "mGhScAHYM5Yiv4Y1O1RASIzYnnA6eRwabfIK2JATlCKhIh0W7bgjr9TTKfRD/oZSBX+/Kv7uw9TA+VqyMRyLI7QbZobMDFlS"
    "g+TCkU0iC2cKct2K0IsAMzmmc8B1tUqS40gVSxdrYAlEkeeOCsMGz91DF5BFFs1Wmjhg1vzgWWSVCpy1sARxCcasZGl1mnH8"
    "XiO5Lo7/c/PgtOBAGb20s0YRnlehh9TcsTOM6y8zWOvi+j+3CWqHqEtwnPTpBvYXrcm6wP7mNNhOxHQBsjvGmiqOMTly9cGS"
    "VEof487dQg8TO/iS1bJ00CoNYDF1/iDQpDliLqQgL86FOJOik4xVUgLPnwqdQgAM+y2aMBbdqo4UkMvCYMELciIJe4U4BaM1"
    "p4GdGo7myxCnEFZrRSjaOiDkHfMNzgnFgQyke9F1Yy5MS/QS0UHyNCU5qobbJXIsAsmxtVlzYRqgRHoG0wTJSGtpjpM4Q5id"
    "vqlNXB7LQX1ubQM0LDCxxoUOOyKdMPR01ki6Q2wvzPUuAuig0k75cAF6Ckb8fCYHdrAOESlpkFwEoon2IUOhKiM8kZfYPoRf"
    "+AiN2TsLns7ibo5xKa7XipxBR7OnyTtXbGW5hJRgrnV1zEdzFpUjiI4mke5Sgv2vH+yf9vkb0/5w/8bnxhyRqEhyOJBGalFi"
    "DKPxQS4xNiD4SwQubbqyBi89J7rQCmKUeGXOhbF/QwylgaQsXRmrEEUmq0xaQJGYRoEnbYDeSsHZPuUTmjfIqThJ2llZKX1U"
    "F2OrLzl+1J80qXJoSTk6b7WLCcxQQ+IVOU+s1kyYQlIPpEpoBjnJYrBVZ1BxOIGr6mxMyITYqVJKWus8R509QgzGAkpSfEB6"
    "Y30dihTmIutQSBN7fjlBZOZs6eTKcXKME9Rk2rj0W5WqlN6qFBtMcYip6FKW0idSRy+I74eiW+LLzxuqM7FQIBI0F/l9wGBH"
    "cOCNLNPxWGMphNgaEGxBgmDEtQKuvTBBn8fkPCNWLUHHEg1IeXRUnAxjfTwMWeT2Rm5TWmCKC6Li+Sd0utqXc6chG6/npm0F"
    "DRgOztdVKRIl8toIETAglMesjhwSjG9Nt0UJJF8WpCb5I9oJjC5jLHhhLGsJgIEeUv5eAYpuR7W+GP5yUfWlGiQYu4ATEcAZ"
    "Ydhnb7aQ8lyDxSU2ntS+xcCDdC4HEdi7AgEOSt3ChnQWqf50Cp3aBS5MI5ROxpUjMSqQ35ETq7nyXksyKFDKCW7r5sSOeZJ8"
    "QY5cObO0Ncvdf//7//tcfGg="
)
print("Delta Drills checker ready — 104 problems. Run dd_check(<problem number>) under any of them.")


<!-- dd:dd-kp-cnn-stride-views -->

## Strides and as_strided views

`cnn.stride-views`


<!-- dd:dd-seg-cnn-stride-views-0 -->

### An axis step is a storage step


A tensor's values live in one flat storage; its shape and strides say how to find them. Shape counts the positions along each axis, and `x.stride()` returns, per axis, how many storage slots to move when that axis's index goes up by one. A fresh `(2, 3)` matrix has strides `(3, 1)`: the next row is three slots on, the next column is one. Its transpose shares the same storage and simply swaps the strides to `(1, 3)` — no value moves.

`x.as_strided(shape, strides)` builds a new view on the same storage from a shape and strides you choose. The rule for choosing them is to derive every stride from the *source's* strides, never from an assumed contiguous layout, because the source may itself be a transposed or sliced view whose strides are not `(n, 1)`. A stride of zero is legal and useful: the index on that axis advances but the address does not, so one row is seen many times without being copied. Every index the new view can reach must land inside the source's storage; `as_strided` does not check for you.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(6).reshape(2,3)\nprint(x.stride(), x.T.stride())\n# Hidden checks\nassert x.stride()==(3,1) and x.T.stride()==(1,3)\n', globals()), end='')


We build a view that shows a vector as two identical rows. The column stride is the vector's own stride, read from `v.stride(0)`; the row stride is zero, so both rows start at the same address.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nv=t.tensor([2.,5.,8.])\nrows=v.as_strided((2,3),(0,v.stride(0)))\nprint(rows)\n# Hidden checks\nassert rows.tolist()==[[2.,5.,8.],[2.,5.,8.]]\n', globals()), end='')




Because the view shares storage, writing to the source shows up in every apparent copy at once — there is only one `5.` in memory.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('v[1]=-1.\nprint(rows)\n# Hidden checks\nassert rows.tolist()==[[2.,-1.,8.],[2.,-1.,8.]]\n', globals()), end='')


<!-- dd:dd-q1217 -->

### Problem 1217 · faded — your turn

Return a transpose view using as_strided, shape (n,m). x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1217)


In [ ]:
#@title 💡 Solution — Problem 1217
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((x.shape[1],x.shape[0]),(x.stride(1),x.stride(0)))


We build a view that shows a vector as two identical rows. The column stride is the vector's own stride, read from `v.stride(0)`; the row stride is zero, so both rows start at the same address.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nv=t.tensor([2.,5.,8.])\nrows=v.as_strided((2,3),(0,v.stride(0)))\nprint(rows)\n# Hidden checks\nassert rows.tolist()==[[2.,5.,8.],[2.,5.,8.]]\n', globals()), end='')




Because the view shares storage, writing to the source shows up in every apparent copy at once — there is only one `5.` in memory.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('v[1]=-1.\nprint(rows)\n# Hidden checks\nassert rows.tolist()==[[2.,-1.,8.],[2.,-1.,8.]]\n', globals()), end='')


<!-- dd:dd-q1218 -->

### Problem 1218 · faded — your turn

Return the main diagonal as a view of length min(m,n), using as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1218)


In [ ]:
#@title 💡 Solution — Problem 1218
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((min(x.shape),),(x.stride(0)+x.stride(1),))


<!-- dd:dd-seg-cnn-stride-views-1 -->

### Build the view, then compute


Many operations are "select some addresses, then reduce". The view does the selecting: a diagonal, for instance, advances one row *and* one column per step, so its single stride is the sum of the two source strides, and its length is the shorter side. The reduction then does the arithmetic on whatever the view exposes. Keeping the two steps separate means the view can be checked by printing it before any number is computed.

The pattern extends to products. A matrix-vector product uses the same vector against every row of the matrix; a zero-stride view of the vector with the matrix's shape expresses that reuse, and `(x * vv).sum(dim=1)` then multiplies and reduces along the feature axis. The reason to reduce over `dim=1` is that the feature axis is the one whose entries were paired; the row axis must survive as one result per row. Treat overlapping views as read-only: a write through the expanded vector would land at one address and change several apparent positions.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(1.,10.).reshape(3,3).T\ndiag=x.as_strided((3,),(x.stride(0)+x.stride(1),))\nprint(diag, diag.sum())\n# Hidden checks\nassert diag.tolist()==[1.,5.,9.] and diag.sum().item()==15\n', globals()), end='')


We compute a matrix-vector product with no `@`. First the view: the vector expanded to the matrix's shape with a zero row stride, so every row of the view is the same vector.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,2.],[3.,4.]])\nv=t.tensor([2.,-1.])\nexpanded=v.as_strided(x.shape,(0,v.stride(0)))\nprint(expanded)\n# Hidden checks\nassert expanded.tolist()==[[2.,-1.],[2.,-1.]]\n', globals()), end='')




Then the arithmetic: multiply elementwise and sum across the feature axis. The result matches `x @ v`, one number per row.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print((x*expanded).sum(dim=1), x@v)\n# Hidden checks\nassert (x*expanded).sum(dim=1).tolist()==[0.,2.] and t.equal((x*expanded).sum(dim=1),x@v)\n', globals()), end='')


<!-- dd:dd-q1219 -->

### Problem 1219 · faded — your turn

Return m copies of v as a view, shape (m,n), using as_strided. x: float (m,n); v: float (n,). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x,v):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1219)


In [ ]:
#@title 💡 Solution — Problem 1219
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,v):
    return v.as_strided(x.shape,(0,v.stride(0)))


We compute a matrix-vector product with no `@`. First the view: the vector expanded to the matrix's shape with a zero row stride, so every row of the view is the same vector.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,2.],[3.,4.]])\nv=t.tensor([2.,-1.])\nexpanded=v.as_strided(x.shape,(0,v.stride(0)))\nprint(expanded)\n# Hidden checks\nassert expanded.tolist()==[[2.,-1.],[2.,-1.]]\n', globals()), end='')




Then the arithmetic: multiply elementwise and sum across the feature axis. The result matches `x @ v`, one number per row.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print((x*expanded).sum(dim=1), x@v)\n# Hidden checks\nassert (x*expanded).sum(dim=1).tolist()==[0.,2.] and t.equal((x*expanded).sum(dim=1),x@v)\n', globals()), end='')


<!-- dd:dd-q1220 -->

### Problem 1220 · faded — your turn

Return x multiplied by v, shape (m,), using only as_strided, elementwise arithmetic and sum. x: float (m,n); v: float (n,). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x,v):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1220)


In [ ]:
#@title 💡 Solution — Problem 1220
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,v):
    vv=v.as_strided(x.shape,(0,v.stride(0)))
    return (x*vv).sum(dim=1)


<!-- dd:dd-q1221 -->

### Problem 1221 · independent

Return the diagonal one step above the main diagonal as a view of length min(m,n-1), using as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1221)


In [ ]:
#@title 💡 Solution — Problem 1221
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    y=x[:,1:]
    return y.as_strided((min(y.shape),),(y.stride(0)+y.stride(1),))


<!-- dd:dd-q1222 -->

### Problem 1222 · independent

Return the outer product of v with itself, shape (n,n), using only as_strided views (stride zero) and elementwise multiplication. v: float (n,). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(v):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1222)


In [ ]:
#@title 💡 Solution — Problem 1222
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(v):
    n=v.shape[0]
    rows=v.as_strided((n,n),(v.stride(0),0))
    cols=v.as_strided((n,n),(0,v.stride(0)))
    return rows*cols


<!-- dd:dd-q1223 -->

### Problem 1223 · independent

Return every other column of x as a view, starting at column zero; use as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1223)


In [ ]:
#@title 💡 Solution — Problem 1223
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((x.shape[0],(x.shape[1]+1)//2),(x.stride(0),2*x.stride(1)))


<!-- dd:dd-q1224 -->

### Problem 1224 · independent

Return a view of shape (m,n,2) whose last axis repeats each source value twice; use as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1224)


In [ ]:
#@title 💡 Solution — Problem 1224
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((x.shape[0],x.shape[1],2),(x.stride(0),x.stride(1),0))


<!-- dd:dd-q1225 -->

### Problem 1225 · independent

Return the sum of the main diagonal as a scalar tensor, using as_strided and sum. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1225)


In [ ]:
#@title 💡 Solution — Problem 1225
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.as_strided((min(x.shape),),(x.stride(0)+x.stride(1),)).sum()


<!-- dd:dd-q1226 -->

### Problem 1226 · independent

Return the row Gram matrix, shape (m,m), using only as_strided, elementwise arithmetic and sum; do not use matmul or einsum. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1226)


In [ ]:
#@title 💡 Solution — Problem 1226
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    m,n=x.shape
    a=x.as_strided((m,m,n),(x.stride(0),0,x.stride(1)))
    b=x.as_strided((m,m,n),(0,x.stride(0),x.stride(1)))
    return (a*b).sum(dim=2)


<!-- dd:dd-q1227 -->

### Problem 1227 · independent

Return sums of consecutive width-two windows from each row, shape (m,n-1). n is at least two. Allowed view operation: as_strided. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1227)


In [ ]:
#@title 💡 Solution — Problem 1227
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    m,n=x.shape
    w=x.as_strided((m,n-1,2),(x.stride(0),x.stride(1),x.stride(1)))
    return w.sum(dim=-1)


<!-- dd:dd-q1228 -->

### Problem 1228 · independent

Return the feature Gram matrix, shape (n,n), using only as_strided, elementwise arithmetic and sum; do not use matmul or einsum. x: float (m,n). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1228)


In [ ]:
#@title 💡 Solution — Problem 1228
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    m,n=x.shape
    a=x.as_strided((m,n,n),(x.stride(0),x.stride(1),0))
    b=x.as_strided((m,n,n),(x.stride(0),0,x.stride(1)))
    return (a*b).sum(dim=0)


<!-- dd:dd-q1229 -->

### Problem 1229 · independent

Return scalar squared length of x multiplied by v, using only as_strided, elementwise arithmetic and sum; do not use matmul or einsum. x: float (m,n); v: float (n,). Inputs may be non-contiguous views with a storage offset.


In [ ]:
import torch as t

def solve(x,v):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1229)


In [ ]:
#@title 💡 Solution — Problem 1229
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,v):
    vv=v.as_strided(x.shape,(0,v.stride(0)))
    y=(x*vv).sum(dim=1)
    return (y*y).sum()


#### Common mistakes

- **A `(m, n)` tensor always has strides `(n, 1)`.** Only when contiguous; a transpose or slice has different strides, and a view built from `(n, 1)` reads the wrong values.
- **A stride of zero copies the data.** It re-reads one address; nothing is allocated.
- **`as_strided` checks that the view fits.** It does not; an out-of-range view reads garbage or crashes.
- **The diagonal stride is `n + 1`.** It is `stride(0) + stride(1)`, which is `n + 1` only for a contiguous matrix.


<!-- dd:dd-kp-cnn-module-state -->

## Modules, parameters and buffers

`cnn.module-state`


<!-- dd:dd-seg-cnn-module-state-0 -->

### A module is a callable computation


PyTorch packages a computation as a class that inherits from `t.nn.Module`. You define one method, `forward(self, x)`, saying how an input becomes an output, and then you call the instance itself — `m(x)` — rather than `m.forward(x)`. The general shape is: `class Name(t.nn.Module):`, then `def forward(self, x):` indented inside it, then `return` the result. `self` is the instance the method was called on; every method of a class receives it first.

The reason to call `m(x)` and not `forward` directly is that the call goes through `Module.__call__`, which runs framework hooks around `forward` (checks, tracing, later on device placement). Calling `forward` skips all of that, so it works today and breaks the day a hook matters. The reason to make even a stateless operation a module is uniformity: a network is a tree of modules, and a tree whose leaves are all the same kind of thing can be printed, moved and saved with one call.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Positive(t.nn.Module):\n    def forward(self,x):\n        return x.clamp(min=0)\nm=Positive()\nprint(m(t.tensor([-3.,2.])))\n# Hidden checks\nassert m(t.tensor([-3.,2.])).tolist()==[0.,2.]\n', globals()), end='')




This is ReLU: positive entries pass through, negative entries become zero. It has no state, which makes it the smallest possible module.


We build a module that halves its input. It needs no `__init__`, because nothing is stored: `forward` is the whole definition. Predict the output before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Half(t.nn.Module):\n    def forward(self,x):\n        return x/2\nm=Half()\nprint(m(t.tensor([3.,-1.])))\n# Hidden checks\nassert m(t.tensor([3.,-1.])).tolist()==[1.5,-0.5]\n', globals()), end='')




The same instance works on any shape, because `forward` only used elementwise arithmetic. Constructing a second instance gives an independent object of the same class; with no state, the two are indistinguishable.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(Half()(t.tensor([[8.,0.]])))\n# Hidden checks\nassert Half()(t.tensor([[8.,0.]])).tolist()==[[4.,0.]]\n', globals()), end='')


<!-- dd:dd-q1169 -->

### Problem 1169 · faded — your turn

Return a parameter-free ReLU module. For any float tensor x, preserve positive entries and replace negative entries with zero.


In [ ]:
import torch as t

def solve():
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1169)


In [ ]:
#@title 💡 Solution — Problem 1169
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve():
    class ReLU(t.nn.Module):
        def forward(self,x):
            return x.clamp(min=0)
    return ReLU()


<!-- dd:dd-seg-cnn-module-state-1 -->

### Parameters are the state an optimizer learns


State that training should change is stored as a `t.nn.Parameter`, assigned to an attribute of `self` inside `__init__`. The procedure has three fixed lines: `def __init__(self):`, then `super().__init__()` as the first statement, then `self.weight = t.nn.Parameter(tensor)`. `forward` reads `self.weight` like any attribute.

`super().__init__()` runs the parent class's constructor — `Module`'s — which creates the bookkeeping that registration relies on. It must come first because the very next line, assigning a `Parameter` to `self`, is intercepted by that bookkeeping: `Module` notices a `Parameter` being attached and records it, so `m.parameters()` can later hand every trainable tensor to an optimizer. Assign before calling `super().__init__()` and there is nothing to record into; the assignment raises. A plain tensor attribute is NOT recorded — it is state the module has but the optimizer cannot see. Wrapping in `Parameter` is what makes the difference, and it also sets `requires_grad=True`, so gradients flow to it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Scale(t.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.weight=t.nn.Parameter(t.tensor(1.5))\n    def forward(self,x):\n        return x*self.weight\nm=Scale()\nprint(m(t.tensor([2.,-4.])))\n# Hidden checks\nassert m(t.tensor([2.,-4.])).detach().tolist()==[3.,-6.]\n', globals()), end='')


We give a module a trainable offset and check that registration actually happened. First the module: the constructor stores one scalar `Parameter` named `bias`, and `forward` adds it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Shift(t.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.bias=t.nn.Parameter(t.tensor(0.25))\n    def forward(self,x):\n        return x+self.bias\nm=Shift()\nprint(m(t.tensor([1.,2.])))\n# Hidden checks\nassert m(t.tensor([1.,2.])).detach().tolist()==[1.25,2.25]\n', globals()), end='')




Now the check that matters for training: `named_parameters()` lists what an optimizer would receive. The name is the attribute name, and the tensor requires a gradient because `Parameter` set that for us.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('names=[name for name,_ in m.named_parameters()]\nprint(names, m.bias.requires_grad)\n# Hidden checks\nassert names==["bias"] and m.bias.requires_grad\n', globals()), end='')


<!-- dd:dd-q1170 -->

### Problem 1170 · faded — your turn

Implement learned scaling: output x multiplied by weight. w: scalar float tensor. Return a new module; register its learnable scalar as weight. Its forward accepts float tensors x of arbitrary shape.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1170)


In [ ]:
#@title 💡 Solution — Problem 1170
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Scale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return x*self.weight
    return Scale()


We give a module a trainable offset and check that registration actually happened. First the module: the constructor stores one scalar `Parameter` named `bias`, and `forward` adds it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Shift(t.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.bias=t.nn.Parameter(t.tensor(0.25))\n    def forward(self,x):\n        return x+self.bias\nm=Shift()\nprint(m(t.tensor([1.,2.])))\n# Hidden checks\nassert m(t.tensor([1.,2.])).detach().tolist()==[1.25,2.25]\n', globals()), end='')




Now the check that matters for training: `named_parameters()` lists what an optimizer would receive. The name is the attribute name, and the tensor requires a gradient because `Parameter` set that for us.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('names=[name for name,_ in m.named_parameters()]\nprint(names, m.bias.requires_grad)\n# Hidden checks\nassert names==["bias"] and m.bias.requires_grad\n', globals()), end='')


<!-- dd:dd-q1172 -->

### Problem 1172 · faded — your turn

Return an affine scalar module with optional bias. w is a scalar float tensor; b is a scalar float tensor or None. Register weight; register bias only when supplied.


In [ ]:
import torch as t

def solve(w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1172)


In [ ]:
#@title 💡 Solution — Problem 1172
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w,b):
    class Affine(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
            self.bias=t.nn.Parameter(b.clone()) if b is not None else None
        def forward(self,x):
            y=x*self.weight
            return y if self.bias is None else y+self.bias
    return Affine()


<!-- dd:dd-seg-cnn-module-state-2 -->

### Buffers are state that is saved but not trained


Some state belongs to a model without being learned: a calibration offset, a fixed mask, and later the running statistics of BatchNorm. Register it with `self.register_buffer("name", tensor)` inside `__init__`, after `super().__init__()`, and read it as `self.name`. The rule for choosing: if an optimizer should update it, `Parameter`; if it must travel with the model but never be trained, buffer; if neither, it is not model state at all.

The reason a buffer is more than a plain attribute is that `Module` tracks it: `m.to(dtype)` and `m.to(device)` convert buffers along with parameters, and `state_dict()` saves them, so a reloaded model has the same offset it was trained with. A plain tensor attribute is left behind by all three. And unlike a `Parameter`, a buffer is not returned by `parameters()`, so the optimizer never touches it — which is exactly the contract.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nclass Offset(t.nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.register_buffer("offset",t.tensor(3.))\n    def forward(self,x):\n        return x+self.offset\nm=Offset()\nprint(m(t.tensor([1.,-1.])), list(m.parameters()))\n# Hidden checks\nassert m(t.tensor([1.,-1.])).tolist()==[4.,2.] and list(m.parameters())==[]\n', globals()), end='')


We confirm the two things a buffer promises: it follows the module through a dtype change, and it appears in the saved state. First the dtype: `m.to(t.float64)` converts the buffer even though it is not a parameter.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('m=Offset().to(t.float64)\nprint(m.offset.dtype)\n# Hidden checks\nassert m.offset.dtype==t.float64\n', globals()), end='')




Then the state dict, which is what `save` writes and `load_state_dict` reads. The buffer is in it under its registered name, and there is no parameter entry because the module has none.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('keys=list(m.state_dict())\nprint(keys)\n# Hidden checks\nassert keys==["offset"]\n', globals()), end='')


<!-- dd:dd-q1171 -->

### Problem 1171 · faded — your turn

Return a calibrated module: subtract fixed scalar b, then multiply by learned scalar w. Store weight as a Parameter and offset as a buffer; both inputs are scalar float tensors.


In [ ]:
import torch as t

def solve(w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1171)


In [ ]:
#@title 💡 Solution — Problem 1171
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w,b):
    class Calibrated(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
            self.register_buffer("offset",b.clone())
        def forward(self,x):
            return (x-self.offset)*self.weight
    return Calibrated()


<!-- dd:dd-q1173 -->

### Problem 1173 · independent

Return a parameter-free module whose forward clips any float tensor x into [0,1]: entries below zero become zero, entries above one become one.


In [ ]:
import torch as t

def solve():
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1173)


In [ ]:
#@title 💡 Solution — Problem 1173
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve():
    class Clip(t.nn.Module):
        def forward(self,x):
            return x.clamp(min=0,max=1)
    return Clip()


<!-- dd:dd-q1174 -->

### Problem 1174 · independent

Return a module that multiplies x by a FIXED scalar taken from w: store it as a buffer named scale, so the module has no parameters.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1174)


In [ ]:
#@title 💡 Solution — Problem 1174
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class FixedScale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.register_buffer("scale",w.clone())
        def forward(self,x):
            return x*self.scale
    return FixedScale()


<!-- dd:dd-q1175 -->

### Problem 1175 · independent

Return a module that adds a trainable scalar offset to x: register the scalar from w as a parameter named bias.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1175)


In [ ]:
#@title 💡 Solution — Problem 1175
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Shift(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.bias=t.nn.Parameter(w.clone())
        def forward(self,x):
            return x+self.bias
    return Shift()


<!-- dd:dd-q1176 -->

### Problem 1176 · independent

Return a module that counts its forward calls in an integer buffer named calls (starting at zero) and returns x unchanged.


In [ ]:
import torch as t

def solve():
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1176)


In [ ]:
#@title 💡 Solution — Problem 1176
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve():
    class Counter(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.register_buffer("calls",t.tensor(0))
        def forward(self,x):
            self.calls += 1
            return x
    return Counter()


<!-- dd:dd-q1177 -->

### Problem 1177 · independent

Return a model containing one child module named scale, whose sole parameter weight starts at scalar tensor w. Output is the positive part of x times that weight. Parent parameter discovery must find scale.weight.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1177)


In [ ]:
#@title 💡 Solution — Problem 1177
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Scale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return x*self.weight
    class Model(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.scale=Scale()
        def forward(self,x):
            return self.scale(x).clamp(min=0)
    return Model()


<!-- dd:dd-q1178 -->

### Problem 1178 · independent

Return a module with a trainable vector parameter weight initialized from w (n,), whose forward returns the weighted sum of x (...,n) along its last axis, shape (...).


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1178)


In [ ]:
#@title 💡 Solution — Problem 1178
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class WeightedSum(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return (x*self.weight).sum(dim=-1)
    return WeightedSum()


<!-- dd:dd-q1179 -->

### Problem 1179 · independent

Return a learned scaling module whose parameter weight (from w) is FROZEN: it stays a registered parameter but requires no gradient. Forward returns x times weight.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1179)


In [ ]:
#@title 💡 Solution — Problem 1179
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Frozen(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone(),requires_grad=False)
        def forward(self,x):
            return x*self.weight
    return Frozen()


<!-- dd:dd-q1180 -->

### Problem 1180 · independent

Return a module using the same trainable scalar on both sides of a ReLU: positive part of x times weight, then multiplied by weight again. Register weight only once. w: scalar float tensor.


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1180)


In [ ]:
#@title 💡 Solution — Problem 1180
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Scale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return (x*self.weight).clamp(min=0)*self.weight
    return Scale()


<!-- dd:dd-q1181 -->

### Problem 1181 · independent

Return a learned scaling module (parameter weight from w, forward x times weight) whose extra_repr reports the current weight as the string "weight=<value>", using the tensor's item().


In [ ]:
import torch as t

def solve(w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1181)


In [ ]:
#@title 💡 Solution — Problem 1181
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(w):
    class Scale(t.nn.Module):
        def __init__(self):
            super().__init__()
            self.weight=t.nn.Parameter(w.clone())
        def forward(self,x):
            return x*self.weight
        def extra_repr(self):
            return "weight="+str(self.weight.item())
    return Scale()


#### Common mistakes

- **Any tensor attribute is trainable.** Only a `t.nn.Parameter` is handed to the optimizer; a plain tensor on `self` is invisible to `parameters()`, `to()` and `state_dict()`.
- **`super().__init__()` is boilerplate that can go anywhere.** It creates the registries; a `Parameter` or buffer assigned before it has nowhere to go and raises.
- **Fixed state can be a plain attribute.** If it must move with the model or be saved, it is a buffer; otherwise it is lost on `to()` and on reload.
- **Using a parameter twice makes two copies.** Reading `self.weight` in two places shares one tensor, and both uses accumulate gradient into it; a second, independent value needs a second `Parameter`.


<!-- dd:dd-kp-cnn-linear-layer -->

## The linear layer

`cnn.linear-layer`


<!-- dd:dd-seg-cnn-linear-layer-0 -->

### Each output owns a weight row


A linear layer turns `in_features` numbers into `out_features` numbers. It stores one weight row per output, so the weight has shape `(out_features, in_features)`, and output `j` is the dot product of row `j` with the input, plus a bias `b[j]`. For a batch `x` of shape `(batch, in_features)` the whole thing is `x @ w.T + b`: transposing makes the shapes `(batch, in) @ (in, out)`, so the feature axes meet and the result is `(batch, out)`.

The reason the weight is stored as `(out, in)` rather than `(in, out)` is convention — PyTorch's `nn.Linear` does it, and every checkpoint you load will — and the transpose in the forward is the price. The reason the bias is `(out_features,)` and simply added is broadcasting: it lines up with the last axis of the product, so each output column gets its own offset on every example. Test shapes with a non-square weight; a square one lets an accidental missing transpose run silently with the wrong numbers.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,2.]])\nw=t.tensor([[2.,0.],[0.,3.],[1.,-1.]])\nb=t.tensor([1.,-1.,0.])\ny=x@w.T+b\nprint(y)\n# Hidden checks\nassert y.tolist()==[[3.,5.,-1.]]\n', globals()), end='')


We run a layer with three inputs and two outputs on a batch of two examples. Row `0` of the weight sums the inputs, row `1` takes the first minus the last, and the bias shifts each output. Predict the `(2, 2)` result before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,1.,1.],[2.,0.,-2.]])\nw=t.tensor([[1.,1.,1.],[1.,0.,-1.]])\nb=t.tensor([0.,10.])\ny=x@w.T+b\nprint(y)\n# Hidden checks\nassert y.tolist()==[[3.,10.],[0.,14.]]\n', globals()), end='')




Drop the transpose and the shapes do not meet: `(2, 3) @ (2, 3)` has no shared inner axis. Catching this at the shape level is the point of a non-square weight.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('try:\n    x@w\nexcept RuntimeError as e:\n    print("shape mismatch:", type(e).__name__)\n# Hidden checks\nassert (x@w.T).shape==(2,2)\n', globals()), end='')


<!-- dd:dd-q1185 -->

### Problem 1185 · faded — your turn

Return the affine output, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1185)


In [ ]:
#@title 💡 Solution — Problem 1185
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y


We run a layer with three inputs and two outputs on a batch of two examples. Row `0` of the weight sums the inputs, row `1` takes the first minus the last, and the bias shifts each output. Predict the `(2, 2)` result before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,1.,1.],[2.,0.,-2.]])\nw=t.tensor([[1.,1.,1.],[1.,0.,-1.]])\nb=t.tensor([0.,10.])\ny=x@w.T+b\nprint(y)\n# Hidden checks\nassert y.tolist()==[[3.,10.],[0.,14.]]\n', globals()), end='')




Drop the transpose and the shapes do not meet: `(2, 3) @ (2, 3)` has no shared inner axis. Catching this at the shape level is the point of a non-square weight.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('try:\n    x@w\nexcept RuntimeError as e:\n    print("shape mismatch:", type(e).__name__)\n# Hidden checks\nassert (x@w.T).shape==(2,2)\n', globals()), end='')


<!-- dd:dd-q1186 -->

### Problem 1186 · faded — your turn

Return the affine output with zero bias, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features).


In [ ]:
import torch as t

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1186)


In [ ]:
#@title 💡 Solution — Problem 1186
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w):
    return x@w.T


<!-- dd:dd-seg-cnn-linear-layer-1 -->

### Leading axes survive; only the last axis is transformed


The layer treats the last axis as features and every axis before it as "which example". A `(batch, seq, in_features)` input — a batch of token sequences, say — goes through the same `x @ w.T + b` and comes out `(batch, seq, out_features)`: the matrix product only touches the last axis of `x`, and the bias broadcasts along it. The general rule is that a linear layer never needs to know how many leading axes there are.

The reason this matters is that it is the same weights for every example and every position. Batching does not make a weight per example; every row of every leading axis is evidence about one shared map, which is what makes the parameters learnable from many examples at once. The weight is the layer's state, created once and reused on every call; `nn.Linear` registers it as a parameter and, when `bias=False`, stores no bias at all rather than a zero one — absent state, not a broadcast constant.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.ones(2,3,4); w=t.ones(5,4)\nprint((x@w.T).shape)\n# Hidden checks\nassert (x@w.T).shape==(2,3,5)\n', globals()), end='')


We push a single vector and a batch of vectors through the same layer and check that the batch result is just the vector results stacked. First the batch: two examples, two inputs, one output.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,0.],[0.,1.]])\nw=t.tensor([[2.,3.]]); b=t.tensor([4.])\nprint(x@w.T+b)\n# Hidden checks\nassert (x@w.T+b).tolist()==[[6.],[7.]]\n', globals()), end='')




Now one example on its own, as a plain `(2,)` vector. The product is `(1,)` — the leading batch axis was never required — and it equals the first row of the batched answer.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(x[0]@w.T+b)\n# Hidden checks\nassert (x[0]@w.T+b).tolist()==[6.]\n', globals()), end='')


<!-- dd:dd-q1187 -->

### Problem 1187 · faded — your turn

Return positive parts of affine outputs, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1187)


In [ ]:
#@title 💡 Solution — Problem 1187
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.clamp(min=0)


We push a single vector and a batch of vectors through the same layer and check that the batch result is just the vector results stacked. First the batch: two examples, two inputs, one output.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[1.,0.],[0.,1.]])\nw=t.tensor([[2.,3.]]); b=t.tensor([4.])\nprint(x@w.T+b)\n# Hidden checks\nassert (x@w.T+b).tolist()==[[6.],[7.]]\n', globals()), end='')




Now one example on its own, as a plain `(2,)` vector. The product is `(1,)` — the leading batch axis was never required — and it equals the first row of the batched answer.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(x[0]@w.T+b)\n# Hidden checks\nassert (x[0]@w.T+b).tolist()==[6.]\n', globals()), end='')


<!-- dd:dd-q1188 -->

### Problem 1188 · faded — your turn

Return the affine output averaged across output features, shape (...). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1188)


In [ ]:
#@title 💡 Solution — Problem 1188
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.mean(dim=-1)


<!-- dd:dd-q1189 -->

### Problem 1189 · independent

Return the affine output of x after centring each example: subtract every example's own mean over in_features first, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1189)


In [ ]:
#@title 💡 Solution — Problem 1189
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    centred=x-x.mean(dim=-1,keepdim=True)
    return centred@w.T+b


<!-- dd:dd-q1190 -->

### Problem 1190 · independent

Return the affine outputs centred to zero mean within each example, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1190)


In [ ]:
#@title 💡 Solution — Problem 1190
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y-y.mean(dim=-1,keepdim=True)


<!-- dd:dd-q1191 -->

### Problem 1191 · independent

Return index of the largest affine output for every example, shape (...); ties choose first. x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1191)


In [ ]:
#@title 💡 Solution — Problem 1191
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.argmax(dim=-1)


<!-- dd:dd-q1192 -->

### Problem 1192 · independent

Return the affine output with all leading example axes combined, shape (number_of_examples,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1192)


In [ ]:
#@title 💡 Solution — Problem 1192
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.reshape(-1,w.shape[0])


<!-- dd:dd-q1193 -->

### Problem 1193 · independent

Return the mean affine output across all examples and positions, shape (out_features,). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1193)


In [ ]:
#@title 💡 Solution — Problem 1193
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.reshape(-1,w.shape[0]).mean(dim=0)


<!-- dd:dd-q1194 -->

### Problem 1194 · independent

Return the tied-weight autoencoder output, shape (...,in_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1194)


In [ ]:
#@title 💡 Solution — Problem 1194
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y.clamp(min=0)@w


<!-- dd:dd-q1195 -->

### Problem 1195 · independent

Return total squared affine activation per example, shape (...). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1195)


In [ ]:
#@title 💡 Solution — Problem 1195
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return (y*y).sum(dim=-1)


<!-- dd:dd-q1196 -->

### Problem 1196 · independent

Return the residual tied-weight autoencoder output: reconstruction plus original input, shape (...,in_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1196)


In [ ]:
#@title 💡 Solution — Problem 1196
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return x+y.clamp(min=0)@w


<!-- dd:dd-q1197 -->

### Problem 1197 · independent

Return the difference between affine outputs on x and on zero input, shape (...,out_features). x: float (...,in_features); w: float (out_features,in_features); b: float (out_features,).


In [ ]:
import torch as t

def solve(x,w,b):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1197)


In [ ]:
#@title 💡 Solution — Problem 1197
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,b):
    y=x@w.T+b
    return y-b


#### Common mistakes

- **The weight is `(in, out)` so that `x @ w` works.** PyTorch stores `(out, in)`; the transpose belongs in the forward.
- **The bias has one entry per example.** It has one per output feature and broadcasts over every leading axis.
- **A batch needs a loop over examples.** One matrix product handles every example and every leading position with the same weights.
- **`bias=False` means a bias of zeros.** It means no bias tensor exists; the state is absent, not constant.


<!-- dd:dd-kp-cnn-mlp -->

## A two-layer MLP for images

`cnn.mlp`


<!-- dd:dd-seg-cnn-mlp-0 -->

### Flatten each image, not the batch


An MLP reads a vector of features per example, but an image arrives as `(channels, height, width)`. The first step is to flatten each image into one vector while keeping examples apart: `x.reshape(x.shape[0], -1)` keeps the batch axis and merges everything after it into a single feature axis of length `channels * height * width`. The `-1` asks reshape to work out that length. The first weight matrix then has shape `(hidden, pixels)`, with `pixels` equal to the number of values in one image, and the hidden pre-activations are `flat @ w1.T + b1`, shape `(batch, hidden)`.

The reason to keep axis `0` is that the batch is not part of any image; flattening it too would glue neighbouring examples into one long vector and make the layer's input width depend on the batch size. The reason the flattening order does not matter for correctness is that the first layer learns a weight per feature position — whatever order the pixels arrive in, it is the same order every time.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(12.).reshape(2,2,3)\nflat=x.reshape(x.shape[0],-1)\nprint(flat)\n# Hidden checks\nassert flat.tolist()==[[0.,1.,2.,3.,4.,5.],[6.,7.,8.,9.,10.,11.]]\n', globals()), end='')


We flatten a batch of three one-channel `2×2` images and feed them to a first layer with five hidden units. Each image has four values, so `pixels = 4` and the weight is `(5, 4)`. Predict the flattened shape before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.ones(3,1,2,2); w1=t.ones(5,4)\nflat=x.reshape(3,-1)\nprint(flat.shape)\n# Hidden checks\nassert flat.shape==(3,4)\n', globals()), end='')




The product keeps the batch of three and produces five hidden values per image. Every input was `1` and every weight was `1`, so each hidden value is the sum of four ones.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('h=flat@w1.T\nprint(h.shape, h[0])\n# Hidden checks\nassert h.shape==(3,5) and h[0].tolist()==[4.,4.,4.,4.,4.]\n', globals()), end='')


<!-- dd:dd-q1201 -->

### Problem 1201 · faded — your turn

Return image features, shape (b,pixels). x: images (b,...), float.


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1201)


In [ ]:
#@title 💡 Solution — Problem 1201
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.reshape(x.shape[0],-1)


We flatten a batch of three one-channel `2×2` images and feed them to a first layer with five hidden units. Each image has four values, so `pixels = 4` and the weight is `(5, 4)`. Predict the flattened shape before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.ones(3,1,2,2); w1=t.ones(5,4)\nflat=x.reshape(3,-1)\nprint(flat.shape)\n# Hidden checks\nassert flat.shape==(3,4)\n', globals()), end='')




The product keeps the batch of three and produces five hidden values per image. Every input was `1` and every weight was `1`, so each hidden value is the sum of four ones.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('h=flat@w1.T\nprint(h.shape, h[0])\n# Hidden checks\nassert h.shape==(3,5) and h[0].tolist()==[4.,4.,4.,4.,4.]\n', globals()), end='')


<!-- dd:dd-q1202 -->

### Problem 1202 · faded — your turn

Return hidden pre-activation values, shape (b,hidden). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1202)


In [ ]:
#@title 💡 Solution — Problem 1202
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1):
    return x.reshape(x.shape[0],-1)@w1.T+b1


<!-- dd:dd-seg-cnn-mlp-1 -->

### A nonlinearity between the layers makes a new representation


Two linear layers in a row, `w2 @ (w1 @ x)`, are together just one linear map, `(w2 @ w1) @ x`; stacking them adds parameters but no expressive power. An activation between them changes that. ReLU, `clamp(min=0)`, zeroes every negative hidden value, so different inputs switch different hidden units on and off and the second layer sees a different linear map in different regions of input space. The full forward is: flatten, first affine, ReLU, second affine.

The reason the last layer has no activation is that its outputs are logits — unnormalized class scores that the loss function consumes directly. A ReLU there would forbid negative scores and a softmax there would double-normalize once the loss applied its own. So the hidden layer gets the nonlinearity and the output layer stays affine. Inspect the hidden activations when debugging: a unit that is zero for every example is a dead unit, and the shape `(batch, hidden)` tells you at once whether the flattening went right.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[-1.,2.],[3.,-2.]])\nw=t.tensor([[1.,1.],[-1.,1.]])\nh=(x@w.T).clamp(min=0)\nprint(h)\n# Hidden checks\nassert h.tolist()==[[1.,3.],[1.,0.]]\n', globals()), end='')


We finish the network on those hidden activations: a second layer with two outputs turns `(batch, hidden)` into `(batch, classes)`. Note that the zeroed hidden unit in the second row contributes nothing, whichever weight multiplies it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nh=t.tensor([[1.,3.],[1.,0.]])\nw2=t.tensor([[1.,-1.],[2.,1.]])\nlogits=h@w2.T\nprint(logits)\n# Hidden checks\nassert logits.tolist()==[[-2.,5.],[1.,2.]]\n', globals()), end='')




Now the same head applied to the hidden values *before* ReLU, to see what the nonlinearity changed. The second row differs, because its negative hidden unit now leaks through.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('raw=t.tensor([[1.,3.],[1.,-5.]])\nprint(raw@w2.T)\n# Hidden checks\nassert (raw@w2.T).tolist()==[[-2.,5.],[6.,-3.]]\n', globals()), end='')


<!-- dd:dd-q1203 -->

### Problem 1203 · faded — your turn

Return hidden activations after the positive-part nonlinearity, shape (b,hidden). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1203)


In [ ]:
#@title 💡 Solution — Problem 1203
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return h


We finish the network on those hidden activations: a second layer with two outputs turns `(batch, hidden)` into `(batch, classes)`. Note that the zeroed hidden unit in the second row contributes nothing, whichever weight multiplies it.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nh=t.tensor([[1.,3.],[1.,0.]])\nw2=t.tensor([[1.,-1.],[2.,1.]])\nlogits=h@w2.T\nprint(logits)\n# Hidden checks\nassert logits.tolist()==[[-2.,5.],[1.,2.]]\n', globals()), end='')




Now the same head applied to the hidden values *before* ReLU, to see what the nonlinearity changed. The second row differs, because its negative hidden unit now leaks through.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('raw=t.tensor([[1.,3.],[1.,-5.]])\nprint(raw@w2.T)\n# Hidden checks\nassert (raw@w2.T).tolist()==[[-2.,5.],[6.,-3.]]\n', globals()), end='')


<!-- dd:dd-q1204 -->

### Problem 1204 · faded — your turn

Return classifier logits, shape (b,classes). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1204)


In [ ]:
#@title 💡 Solution — Problem 1204
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    return y


<!-- dd:dd-q1205 -->

### Problem 1205 · independent

Return the logits of the two-layer image classifier averaged over the batch, shape (classes,). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1205)


In [ ]:
#@title 💡 Solution — Problem 1205
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return (h@w2.T+b2).mean(dim=0)


<!-- dd:dd-q1206 -->

### Problem 1206 · independent

Return predicted class indices, shape (b,); ties choose first. x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1206)


In [ ]:
#@title 💡 Solution — Problem 1206
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    return y.argmax(dim=1)


<!-- dd:dd-q1207 -->

### Problem 1207 · independent

Return mean hidden activation over the batch, shape (hidden,). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1207)


In [ ]:
#@title 💡 Solution — Problem 1207
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return h.mean(dim=0)


<!-- dd:dd-q1208 -->

### Problem 1208 · independent

Return number of active hidden features per example, shape (b,); positive values are active. x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1208)


In [ ]:
#@title 💡 Solution — Problem 1208
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return (h>0).sum(dim=1)


<!-- dd:dd-q1209 -->

### Problem 1209 · independent

Return logits with the first hidden feature silenced, shape (b,classes). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1209)


In [ ]:
#@title 💡 Solution — Problem 1209
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    h[:,0]=0
    return h@w2.T+b2


<!-- dd:dd-q1210 -->

### Problem 1210 · independent

Return class probabilities, shape (b,classes). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1210)


In [ ]:
#@title 💡 Solution — Problem 1210
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    z=y-y.max(dim=1,keepdim=True)[0]
    e=z.exp()
    return e/e.sum(dim=1,keepdim=True)


<!-- dd:dd-q1211 -->

### Problem 1211 · independent

Return the change in logits caused by removing the hidden nonlinearity, shape (b,classes): linear-only minus normal model. x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1211)


In [ ]:
#@title 💡 Solution — Problem 1211
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    plain=(flat@w1.T+b1)@w2.T+b2
    return plain-y


<!-- dd:dd-q1212 -->

### Problem 1212 · independent

Return the first hidden feature’s contribution to every logit, shape (b,classes), excluding output bias. x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1212)


In [ ]:
#@title 💡 Solution — Problem 1212
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    return h[:,0,None]*w2[None,:,0]


<!-- dd:dd-q1213 -->

### Problem 1213 · independent

Return logits relative to each example’s first class score, shape (b,classes). x: images (b,...), float; w1: (hidden,pixels); b1: (hidden,); w2: (classes,hidden); b2: (classes,). pixels is the number of values in one image.


In [ ]:
import torch as t

def solve(x,w1,b1,w2,b2):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1213)


In [ ]:
#@title 💡 Solution — Problem 1213
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w1,b1,w2,b2):
    flat=x.reshape(x.shape[0],-1)
    h=(flat@w1.T+b1).clamp(min=0)
    y=h@w2.T+b2
    return y-y[:,0,None]


#### Common mistakes

- **`x.reshape(-1)` flattens the images.** It flattens the whole batch into one vector; the batch axis must be kept.
- **Two linear layers learn more than one.** Without a nonlinearity between them they compose into a single linear map.
- **The output layer needs ReLU or softmax.** Logits are consumed raw by the loss; an activation there changes the model.
- **`pixels` is `height * width`.** It is every value in one image, channels included.


<!-- dd:dd-kp-cnn-convolution-1d -->

## 1-D convolution from strided windows

`cnn.convolution-1d`


<!-- dd:dd-seg-cnn-convolution-1d-0 -->

### One filter, many windows


A convolution scores every local patch of a signal with the same small filter. For a 1-D signal of width `W` and a kernel of length `K`, there are `W - K + 1` valid windows — every run of `K` consecutive values — and output `p` is the dot product of window `p` with the filter. The procedure is: build a view whose rows are the windows, multiply each row by the filter, sum along the kernel axis. The view is `x.as_strided((W-K+1, K), (stride, stride))`: moving to the next window and moving to the next value inside a window are both one storage step, so the two strides are equal, and no window is ever copied.

With channels and a batch the same idea gains axes. Input `x` is `(batch, in_channels, width)`, the filter bank `w` is `(out_channels, in_channels, kernel)`, and the window view is `(batch, in_channels, positions, kernel)`. Each output filter reads every input channel, so the einsum sums over the channel *and* kernel axes and keeps batch, output channel and position: `"b c p k, o c k -> b o p"`. The reason PyTorch's "convolution" does not flip the filter is that a learned filter can be learned flipped; the first weight meets the first value of each window, which is what the strided view gives for free.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([1.,2.,4.,8.])\nw=t.tensor([1.,-1.])\nwindows=x.as_strided((3,2),(1,1))\nprint((windows*w).sum(dim=1))\n# Hidden checks\nassert (windows*w).sum(dim=1).tolist()==[-1.,-2.,-4.]\n', globals()), end='')


We convolve one example with two input channels and a single output filter. First the window view: width `3` and kernel `2` give two positions, and the strides come from `x.stride()` rather than from assumptions.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport einops\nx=t.tensor([[[1.,2.,3.],[4.,5.,6.]]])\nw=t.tensor([[[1.,0.],[0.,1.]]])\nbs,cs,ws=x.stride()\nwindows=x.as_strided((1,2,2,2),(bs,cs,ws,ws))\nprint(windows[0,0], windows[0,1])\n# Hidden checks\nassert windows[0,0].tolist()==[[1.,2.],[2.,3.]] and windows[0,1].tolist()==[[4.,5.],[5.,6.]]\n', globals()), end='')




The filter takes the first value of channel `0` and the second value of channel `1`, so position `0` gives `1 + 5` and position `1` gives `2 + 6`. The einsum sums over `c` and `k` and keeps `b o p`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('y=einops.einsum(windows,w,"b c p k, o c k -> b o p")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[6.,8.]]]\n', globals()), end='')


<!-- dd:dd-q1233 -->

### Problem 1233 · faded — your turn

Return valid convolution with stride one and no padding. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1233)


In [ ]:
#@title 💡 Solution — Problem 1233
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y


We convolve one example with two input channels and a single output filter. First the window view: width `3` and kernel `2` give two positions, and the strides come from `x.stride()` rather than from assumptions.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport einops\nx=t.tensor([[[1.,2.,3.],[4.,5.,6.]]])\nw=t.tensor([[[1.,0.],[0.,1.]]])\nbs,cs,ws=x.stride()\nwindows=x.as_strided((1,2,2,2),(bs,cs,ws,ws))\nprint(windows[0,0], windows[0,1])\n# Hidden checks\nassert windows[0,0].tolist()==[[1.,2.],[2.,3.]] and windows[0,1].tolist()==[[4.,5.],[5.,6.]]\n', globals()), end='')




The filter takes the first value of channel `0` and the second value of channel `1`, so position `0` gives `1 + 5` and position `1` gives `2 + 6`. The einsum sums over `c` and `k` and keeps `b o p`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('y=einops.einsum(windows,w,"b c p k, o c k -> b o p")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[6.,8.]]]\n', globals()), end='')


<!-- dd:dd-q1234 -->

### Problem 1234 · faded — your turn

Return each filter’s valid output averaged over positions. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1234)


In [ ]:
#@title 💡 Solution — Problem 1234
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y.mean(dim=2)


<!-- dd:dd-seg-cnn-convolution-1d-1 -->

### Stride chooses positions; padding extends the signal


Two knobs change which windows exist. A stride `S` makes consecutive windows start `S` values apart instead of one, so the view's position stride becomes `S * stride` while the kernel stride stays `stride`. Padding `P` adds `P` known values — zeros, for convolution — at each end of the signal before any window is taken, so windows can be centred on the edges. With both, the output width is `1 + (W + 2P - K) // S`: the number of window starts that fit.

The reason padding must be a real, allocated tensor is that a view cannot address storage that does not exist; so make a zero tensor of the padded shape, write `x` into its middle with a slice assignment, and take the strides from the *padded* tensor, since that is what the windows read. The reason zero is the right padding value here is that it is neutral for the weighted sum: a zero times any weight adds nothing. It is not neutral for every operation — a maximum over negative values would be wrongly beaten by a zero — which the pooling lesson handles differently.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([1.,2.,3.])\npadded=t.zeros(5)\npadded[1:4]=x\nprint(padded)\n# Hidden checks\nassert padded.tolist()==[0.,1.,2.,3.,0.]\n', globals()), end='')


We take stride-2 windows of kernel `3` over that padded signal. The formula gives `1 + (5 - 3) // 2 = 2` windows, and the position stride is `2 * 1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\npadded=t.tensor([0.,1.,2.,3.,0.])\nwindows=padded.as_strided((2,3),(2*padded.stride(0),padded.stride(0)))\nprint(windows)\n# Hidden checks\nassert windows.tolist()==[[0.,1.,2.],[2.,3.,0.]]\n', globals()), end='')




Both windows include a padding zero, which contributes nothing to the sum. With a filter of all ones, each output is the sum of the real values it covers.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('w=t.ones(3)\nprint((windows*w).sum(dim=1))\n# Hidden checks\nassert (windows*w).sum(dim=1).tolist()==[3.,5.]\n', globals()), end='')


<!-- dd:dd-q1235 -->

### Problem 1235 · faded — your turn

Return x with p zeros added at both ends of its width axis, preserving batch and channels. x: float (batch,in_channels,width); p: zeros added at each end.


In [ ]:
import torch as t

def solve(x,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1235)


In [ ]:
#@title 💡 Solution — Problem 1235
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,p):
    b,c,width=x.shape
    y=t.zeros((b,c,width+2*p),dtype=x.dtype)
    y[:,:,p:p+width]=x
    return y


We take stride-2 windows of kernel `3` over that padded signal. The formula gives `1 + (5 - 3) // 2 = 2` windows, and the position stride is `2 * 1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\npadded=t.tensor([0.,1.,2.,3.,0.])\nwindows=padded.as_strided((2,3),(2*padded.stride(0),padded.stride(0)))\nprint(windows)\n# Hidden checks\nassert windows.tolist()==[[0.,1.,2.],[2.,3.,0.]]\n', globals()), end='')




Both windows include a padding zero, which contributes nothing to the sum. With a filter of all ones, each output is the sum of the real values it covers.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('w=t.ones(3)\nprint((windows*w).sum(dim=1))\n# Hidden checks\nassert (windows*w).sum(dim=1).tolist()==[3.,5.]\n', globals()), end='')


<!-- dd:dd-q1236 -->

### Problem 1236 · faded — your turn

Return convolution with stride s and symmetric zero padding p. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1236)


In [ ]:
#@title 💡 Solution — Problem 1236
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y


<!-- dd:dd-q1237 -->

### Problem 1237 · independent

Return the valid convolution (stride one, no padding) summed over output channels, shape (batch,out_width). x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1237)


In [ ]:
#@title 💡 Solution — Problem 1237
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    n=width-k+1
    step=x.stride()
    win=x.as_strided((b,c,n,k),(step[0],step[1],step[2],step[2]))
    return einops.einsum(win,w,"b c p k, o c k -> b p")


<!-- dd:dd-q1238 -->

### Problem 1238 · independent

Return valid convolution followed by positive-part activation. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1238)


In [ ]:
#@title 💡 Solution — Problem 1238
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y.clamp(min=0)


<!-- dd:dd-q1239 -->

### Problem 1239 · independent

Return each filter’s maximum valid activation over positions. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel).


In [ ]:
import torch as t
import einops

def solve(x,w):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1239)


In [ ]:
#@title 💡 Solution — Problem 1239
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y.max(dim=2)[0]


<!-- dd:dd-q1240 -->

### Problem 1240 · independent

Return valid convolution with stride s and no padding. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1240)


In [ ]:
#@title 💡 Solution — Problem 1240
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s):
    b,c,width=x.shape
    o,_,k=w.shape
    win=x.as_strided((b,c,width-k+1,k),(x.stride(0),x.stride(1),x.stride(2),x.stride(2)))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y[:,:,::s]


<!-- dd:dd-q1241 -->

### Problem 1241 · independent

Return the output width of the convolution with stride s and padding p, as an integer. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1241)


In [ ]:
#@title 💡 Solution — Problem 1241
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,s,p):
    return 1+(x.shape[2]+2*p-w.shape[2])//s


<!-- dd:dd-q1242 -->

### Problem 1242 · independent

Return convolution responses to channel-centred filters: each kernel position has zero mean across input channels. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1242)


In [ ]:
#@title 💡 Solution — Problem 1242
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    w=w-w.mean(dim=1,keepdim=True)
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y


<!-- dd:dd-q1243 -->

### Problem 1243 · independent

Return each filter’s activation averaged across both batch and positions, using stride s and padding p. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1243)


In [ ]:
#@title 💡 Solution — Problem 1243
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return y.mean(dim=(0,2))


<!-- dd:dd-q1244 -->

### Problem 1244 · independent

Return the convolution response summed over the batch, shape (out_channels,out_width), with stride s and padding p. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1244)


In [ ]:
#@title 💡 Solution — Problem 1244
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    return einops.einsum(win,w,"b c p k, o c k -> o p")


<!-- dd:dd-q1245 -->

### Problem 1245 · independent

Return filters ranked by descending mean squared activation over batch and positions. Use stride s and padding p; scores are distinct. x: float (batch,in_channels,width); w: float (out_channels,in_channels,kernel); s: stride ≥ 1; p: zeros added at each end. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1245)


In [ ]:
#@title 💡 Solution — Problem 1245
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,width=x.shape
    o,_,k=w.shape
    xx=t.zeros((b,c,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+width]=x
    step=xx.stride()
    n=1+(xx.shape[2]-k)//s
    win=xx.as_strided((b,c,n,k),(step[0],step[1],s*step[2],step[2]))
    y=einops.einsum(win,w,"b c p k, o c k -> b o p")
    return t.argsort((y*y).mean(dim=(0,2)),descending=True)


#### Common mistakes

- **The filter is reversed before multiplying.** Not in PyTorch; weight `0` meets the first value of each window.
- **Each output channel reads one input channel.** It reads all of them; the sum runs over channel and kernel axes together.
- **Padding is a view.** Padding allocates a new tensor; strides must be read from that tensor, not from `x`.
- **Stride `S` skips values inside a window.** It skips window *starts*; each window still holds `K` consecutive values.


<!-- dd:dd-kp-cnn-convolution-2d -->

## 2-D convolution

`cnn.convolution-2d`


<!-- dd:dd-seg-cnn-convolution-2d-0 -->

### An image window has two spatial axes


A 2-D convolution is the 1-D one with a second spatial axis everywhere. The input is `(batch, in_channels, height, width)`, the filter bank is `(out_channels, in_channels, kh, kw)`, and the window view has six axes: batch, input channel, output row, output column, kernel row, kernel column. Output rows number `1 + (H + 2P - kh) // S` and output columns `1 + (W + 2P - kw) // S`. The view's strides are read from the (padded) input: `(bs, cs, S*hs, S*ws, hs, ws)` — moving one output row is `S` image rows, moving one kernel row is one image row, and likewise for columns. The einsum sums over input channel and both kernel axes: `"b c h w i j, o c i j -> b o h w"`.

The reason to keep six named axes rather than flattening is that every mistake in a convolution is an axis mistake — rows and columns swapped, stride applied to the kernel axis, channels summed at the wrong place — and names make each one visible. The reason to test with rectangular images and rectangular kernels is the same: a `3×3` kernel on a square image hides a row/column swap; a `2×3` kernel on a `3×5` image does not. Padding, as in 1-D, is an allocated zero tensor with `x` written into its interior on both spatial axes.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport einops\nx=t.arange(1.,10.).reshape(1,1,3,3)\nwindows=x.as_strided((1,1,2,2,2,2),(9,9,3,1,3,1))\nw=t.ones(1,1,2,2)\ny=einops.einsum(windows,w,"b c h w i j, o c i j -> b o h w")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[[12.,16.],[24.,28.]]]]\n', globals()), end='')


We build the window view for a rectangular image and a rectangular kernel, reading strides from the tensor. A `3×5` image with a `2×3` kernel at stride `1` gives `2` output rows and `3` output columns.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(1.,16.).reshape(1,1,3,5)\nbs,cs,hs,ws=x.stride()\nwin=x.as_strided((1,1,2,3,2,3),(bs,cs,hs,ws,hs,ws))\nprint(win.shape, win[0,0,1,2])\n# Hidden checks\nassert win.shape==(1,1,2,3,2,3) and win[0,0,1,2].tolist()==[[8.,9.,10.],[13.,14.,15.]]\n', globals()), end='')




Window `(1, 2)` is the bottom-right patch, and a kernel of ones sums it. The full output has one such sum per window position; predict the corner values before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import einops\ny=einops.einsum(win,t.ones(1,1,2,3),"b c h w i j, o c i j -> b o h w")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[[27.,33.,39.],[57.,63.,69.]]]]\n', globals()), end='')


<!-- dd:dd-q1251 -->

### Problem 1251 · faded — your turn

Return x with p zeros added on every side of both spatial axes. x: float (batch,in_channels,height,width); p: zero padding on every side.


In [ ]:
import torch as t

def solve(x,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1251)


In [ ]:
#@title 💡 Solution — Problem 1251
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,p):
    b,c,h,width=x.shape
    y=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    y[:,:,p:p+h,p:p+width]=x
    return y


We build the window view for a rectangular image and a rectangular kernel, reading strides from the tensor. A `3×5` image with a `2×3` kernel at stride `1` gives `2` output rows and `3` output columns.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(1.,16.).reshape(1,1,3,5)\nbs,cs,hs,ws=x.stride()\nwin=x.as_strided((1,1,2,3,2,3),(bs,cs,hs,ws,hs,ws))\nprint(win.shape, win[0,0,1,2])\n# Hidden checks\nassert win.shape==(1,1,2,3,2,3) and win[0,0,1,2].tolist()==[[8.,9.,10.],[13.,14.,15.]]\n', globals()), end='')




Window `(1, 2)` is the bottom-right patch, and a kernel of ones sums it. The full output has one such sum per window position; predict the corner values before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import einops\ny=einops.einsum(win,t.ones(1,1,2,3),"b c h w i j, o c i j -> b o h w")\nprint(y)\n# Hidden checks\nassert y.tolist()==[[[[27.,33.,39.],[57.,63.,69.]]]]\n', globals()), end='')


<!-- dd:dd-q1249 -->

### Problem 1249 · faded — your turn

Return the image convolution with stride s and padding p. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1249)


In [ ]:
#@title 💡 Solution — Problem 1249
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y


<!-- dd:dd-seg-cnn-convolution-2d-1 -->

### The library function performs the same convolution


`torch.nn.functional.conv2d(x, w, stride=s, padding=p)` — imported as `F.conv2d` — computes exactly the windowed einsum: same axis conventions, same output size formula, zero padding on every side, no filter flip. It takes the weight as an argument on every call, which is what makes it a *function*: it owns nothing. A `Conv2d` module is the thin wrapper that stores the weight as a trainable `Parameter` and calls the function with it; ARENA's version omits the bias because a BatchNorm that follows supplies its own offset.

The reason to write the strided version first and then switch to `F.conv2d` is verification: the library call is the oracle your hand-built windows are checked against, and once they agree you understand what the fast path does. The reason fan-in matters is initialization: each output filter reads `in_channels * kh * kw` values, and the weight scale must shrink as that count grows so outputs do not blow up — the same rule as `in_features` for a linear layer. The number of output channels sets how many filters are learned, not how many values each one reads.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport torch.nn.functional as F\nx=t.arange(1.,16.).reshape(1,1,3,5)\nw=t.ones(1,1,2,3)\nprint(F.conv2d(x,w).shape)\n# Hidden checks\nassert F.conv2d(x,w).shape==(1,1,2,3)\n', globals()), end='')


We run the same ones-kernel over a ones-image with stride `2` and padding `1`, where the padding shows up as smaller sums at the edges. A `3×5` image padded to `5×7` with a `3×3` kernel at stride `2` gives `2` rows and `3` columns.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport torch.nn.functional as F\nx=t.ones(1,1,3,5)\nw=t.ones(1,1,3,3)\ny=F.conv2d(x,w,stride=2,padding=1)\nprint(y.shape)\n# Hidden checks\nassert y.shape==(1,1,2,3)\n', globals()), end='')




Corner windows overlap the padding on two sides and see four real pixels; edge windows see six. The interior of a bigger image would see nine.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(y)\n# Hidden checks\nassert y.tolist()==[[[[4.,6.,4.],[4.,6.,4.]]]]\n', globals()), end='')


<!-- dd:dd-q1252 -->

### Problem 1252 · faded — your turn

Return the image convolution through the functional library API. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import torch.nn.functional as F

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1252)


In [ ]:
#@title 💡 Solution — Problem 1252
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import torch.nn.functional as F

def solve(x,w,s,p):
    return F.conv2d(x,w,stride=s,padding=p)


We run the same ones-kernel over a ones-image with stride `2` and padding `1`, where the padding shows up as smaller sums at the edges. A `3×5` image padded to `5×7` with a `3×3` kernel at stride `2` gives `2` rows and `3` columns.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nimport torch.nn.functional as F\nx=t.ones(1,1,3,5)\nw=t.ones(1,1,3,3)\ny=F.conv2d(x,w,stride=2,padding=1)\nprint(y.shape)\n# Hidden checks\nassert y.shape==(1,1,2,3)\n', globals()), end='')




Corner windows overlap the padding on two sides and see four real pixels; edge windows see six. The interior of a bigger image would see nine.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(y)\n# Hidden checks\nassert y.tolist()==[[[[4.,6.,4.],[4.,6.,4.]]]]\n', globals()), end='')


<!-- dd:dd-q1250 -->

### Problem 1250 · faded — your turn

Return the image convolution followed by ReLU. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1250)


In [ ]:
#@title 💡 Solution — Problem 1250
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y.clamp(min=0)


<!-- dd:dd-q1253 -->

### Problem 1253 · independent

Return the output height and width of the image convolution with stride s and padding p, as a tuple of two integers. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1253)


In [ ]:
#@title 💡 Solution — Problem 1253
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,w,s,p):
    return (1+(x.shape[2]+2*p-w.shape[2])//s, 1+(x.shape[3]+2*p-w.shape[3])//s)


<!-- dd:dd-q1254 -->

### Problem 1254 · independent

Return average convolution activation per example and output channel, shape (batch,out_channels). x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1254)


In [ ]:
#@title 💡 Solution — Problem 1254
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y.mean(dim=(2,3))


<!-- dd:dd-q1255 -->

### Problem 1255 · independent

Return image convolution with each filter adjusted to have zero total coefficient. Use stride s and padding p. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1255)


In [ ]:
#@title 💡 Solution — Problem 1255
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    w=w-w.mean(dim=(1,2,3),keepdim=True)
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y


<!-- dd:dd-q1256 -->

### Problem 1256 · independent

Return each output channel’s maximum response over all examples and positions, shape (out_channels,). x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1256)


In [ ]:
#@title 💡 Solution — Problem 1256
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y.max(dim=3)[0].max(dim=2)[0].max(dim=0)[0]


<!-- dd:dd-q1257 -->

### Problem 1257 · independent

Return convolution activation at the upper-left output position, shape (batch,out_channels). Include padding exactly as in the full operation. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1257)


In [ ]:
#@title 💡 Solution — Problem 1257
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y[:,:,0,0]


<!-- dd:dd-q1258 -->

### Problem 1258 · independent

Return convolution feature maps centred to zero spatial mean independently for each example and output channel. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1258)


In [ ]:
#@title 💡 Solution — Problem 1258
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y-y.mean(dim=(2,3),keepdim=True)


<!-- dd:dd-q1259 -->

### Problem 1259 · independent

Return shape (batch,out_channels,2), containing each response map’s spatial minimum followed by its maximum. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1259)


In [ ]:
#@title 💡 Solution — Problem 1259
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    low=y.min(dim=3)[0].min(dim=2)[0]
    high=y.max(dim=3)[0].max(dim=2)[0]
    return t.stack((low,high),dim=-1)


<!-- dd:dd-q1260 -->

### Problem 1260 · independent

Return per-example total energy in each output channel, divided by total energy over all channels. Zero-energy examples return zeros. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1260)


In [ ]:
#@title 💡 Solution — Problem 1260
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    energy=(y*y).sum(dim=(2,3))
    total=energy.sum(dim=1,keepdim=True)
    return energy/t.where(total>0,total,t.ones_like(total))


<!-- dd:dd-q1261 -->

### Problem 1261 · independent

Return the response of locally mean-centred image patches to the filters. Patch means include padding and all input channels. x: float (batch,in_channels,height,width); w: float (out_channels,in_channels,kh,kw); s: stride ≥ 1; p: zero padding on every side. The kernel fits the padded input.


In [ ]:
import torch as t
import einops

def solve(x,w,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1261)


In [ ]:
#@title 💡 Solution — Problem 1261
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t
import einops

def solve(x,w,s,p):
    b,c,h,width=x.shape
    o,_,kh,kw=w.shape
    xx=t.zeros((b,c,h+2*p,width+2*p),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+width]=x
    bs,cs,hs,ws=xx.stride()
    oh=1+(h+2*p-kh)//s
    ow=1+(width+2*p-kw)//s
    win=xx.as_strided((b,c,oh,ow,kh,kw),(bs,cs,s*hs,s*ws,hs,ws))
    win=win-win.mean(dim=(1,4,5),keepdim=True)
    y=einops.einsum(win,w,"b c h w i j, o c i j -> b o h w")
    return y


#### Common mistakes

- **Stride scales the kernel strides too.** Only the output-position strides are multiplied by `S`; kernel strides stay one image step.
- **A square test image is enough.** It hides a row/column swap; use rectangular images and kernels.
- **`F.conv2d` flips the kernel.** It does not; it is the windowed sum with the weight as given.
- **More output channels means a bigger fan-in.** Fan-in is `in_channels * kh * kw`; output channels only set how many filters exist.


<!-- dd:dd-kp-cnn-pooling -->

## Pooling

`cnn.pooling`


<!-- dd:dd-seg-cnn-pooling-0 -->

### A spatial summary keeps examples and channels apart


Global average pooling replaces each channel's whole image with a single number — its mean over height and width. On `(batch, channels, height, width)` that is `x.mean(dim=(2, 3))`, and the result is `(batch, channels)`: one row per example, one value per channel, ready for a linear classifier head. The reduction names the two spatial axes and nothing else, so examples are never averaged together and channels are never mixed.

The reason to reduce only the spatial axes is what each axis means. Averaging over the batch would blend different images; averaging over channels would blend different feature detectors. The reason the mean is the usual choice at the end of a network is that it asks "how much of this feature appeared across the image", which is robust to where it appeared; a global maximum instead asks "did this feature appear strongly anywhere". Neither has a learned weight — pooling is a fixed summary, which is why it costs nothing to train.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[1.,3.],[5.,7.]],[[2.,2.],[4.,4.]]]])\nprint(x.mean(dim=(2,3)))\n# Hidden checks\nassert x.mean(dim=(2,3)).tolist()==[[4.,3.]]\n', globals()), end='')


We pool a batch of two one-channel images: one flat, one with a single bright pixel. The mean sees the total brightness; the maximum sees the peak. Predict both before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[2.,2.],[2.,2.]]],[[[0.,0.],[0.,8.]]]])\nprint(x.mean(dim=(2,3)))\n# Hidden checks\nassert x.mean(dim=(2,3)).tolist()==[[2.],[2.]]\n', globals()), end='')




The two images have the same mean but very different maxima. Reducing `dim=3` then `dim=2` takes the max over columns and then rows, and `[0]` picks the values rather than the indices.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(x.max(dim=3)[0].max(dim=2)[0])\n# Hidden checks\nassert x.max(dim=3)[0].max(dim=2)[0].tolist()==[[2.],[8.]]\n', globals()), end='')


<!-- dd:dd-q1265 -->

### Problem 1265 · faded — your turn

Return global spatial average, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1265)


In [ ]:
#@title 💡 Solution — Problem 1265
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.mean(dim=(2,3))


We pool a batch of two one-channel images: one flat, one with a single bright pixel. The mean sees the total brightness; the maximum sees the peak. Predict both before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[2.,2.],[2.,2.]]],[[[0.,0.],[0.,8.]]]])\nprint(x.mean(dim=(2,3)))\n# Hidden checks\nassert x.mean(dim=(2,3)).tolist()==[[2.],[2.]]\n', globals()), end='')




The two images have the same mean but very different maxima. Reducing `dim=3` then `dim=2` takes the max over columns and then rows, and `[0]` picks the values rather than the indices.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(x.max(dim=3)[0].max(dim=2)[0])\n# Hidden checks\nassert x.max(dim=3)[0].max(dim=2)[0].tolist()==[[2.],[8.]]\n', globals()), end='')


<!-- dd:dd-q1268 -->

### Problem 1268 · faded — your turn

Return each channel’s spatial maximum, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1268)


In [ ]:
#@title 💡 Solution — Problem 1268
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.max(dim=3)[0].max(dim=2)[0]


<!-- dd:dd-seg-cnn-pooling-1 -->

### Local maxima need a neutral boundary


Local max pooling keeps coarse spatial layout: each output is the largest value in one `k×k` window, with windows placed by a stride, exactly as convolution places them. The window view is the six-axis one from convolution, `(batch, channels, out_h, out_w, k, k)`, and the reduction is a maximum over the last two axes — `win.max(dim=5)[0].max(dim=4)[0]` — so channels are never combined. Output size follows the same `1 + (H + 2P - k) // S` rule.

The reason padding must be `-inf` here, not zero, is that the maximum is not neutral to zero: a window lying entirely over negative activations should return its largest negative value, and a zero pad would wrongly win. `t.full(shape, float("-inf"))` makes a boundary that can never be the maximum, and writing `x` into its interior gives the padded tensor whose strides the view reads. The reason max pooling is used between convolutions is that it halves the resolution while keeping the strongest response in each neighbourhood, so later layers see a wider context per pixel.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([-4.,-2.])\npadded=t.full((4,),float("-inf"))\npadded[1:3]=x\nprint(padded.as_strided((3,2),(1,1)).max(dim=1)[0])\n# Hidden checks\nassert padded.as_strided((3,2),(1,1)).max(dim=1)[0].tolist()==[-4.,-2.,-2.]\n', globals()), end='')


We pool a `4×4` image with `2×2` windows at stride `2`, which tiles it into four blocks. The view's output-position strides are twice the image strides; the kernel strides are the image strides themselves.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(16.).reshape(1,1,4,4)\nbs,cs,hs,ws=x.stride()\nwin=x.as_strided((1,1,2,2,2,2),(bs,cs,2*hs,2*ws,hs,ws))\nprint(win[0,0,0,1])\n# Hidden checks\nassert win[0,0,0,1].tolist()==[[2.,3.],[6.,7.]]\n', globals()), end='')




Each block's maximum is its bottom-right value, because the image increases along both axes. The result is a `2×2` map: the same layout, half the resolution.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(win.max(dim=5)[0].max(dim=4)[0])\n# Hidden checks\nassert win.max(dim=5)[0].max(dim=4)[0].tolist()==[[[[5.,7.],[13.,15.]]]]\n', globals()), end='')


<!-- dd:dd-q1266 -->

### Problem 1266 · faded — your turn

Return local max pooling with stride s and padding p. x: float (batch,channels,height,width); k: square kernel size; s: stride; p: zero padding on every side, 0 ≤ p ≤ k//2. The kernel fits the padded image.


In [ ]:
import torch as t

def solve(x,k,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1266)


In [ ]:
#@title 💡 Solution — Problem 1266
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,k,s,p):
    b,c,h,w=x.shape
    xx=t.full((b,c,h+2*p,w+2*p),float("-inf"),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+w]=x
    bs,cs,hs,ws=xx.stride()
    win=xx.as_strided((b,c,1+(h+2*p-k)//s,1+(w+2*p-k)//s,k,k),(bs,cs,s*hs,s*ws,hs,ws))
    y=win.max(dim=5)[0].max(dim=4)[0]
    return y


We pool a `4×4` image with `2×2` windows at stride `2`, which tiles it into four blocks. The view's output-position strides are twice the image strides; the kernel strides are the image strides themselves.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.arange(16.).reshape(1,1,4,4)\nbs,cs,hs,ws=x.stride()\nwin=x.as_strided((1,1,2,2,2,2),(bs,cs,2*hs,2*ws,hs,ws))\nprint(win[0,0,0,1])\n# Hidden checks\nassert win[0,0,0,1].tolist()==[[2.,3.],[6.,7.]]\n', globals()), end='')




Each block's maximum is its bottom-right value, because the image increases along both axes. The result is a `2×2` map: the same layout, half the resolution.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('print(win.max(dim=5)[0].max(dim=4)[0])\n# Hidden checks\nassert win.max(dim=5)[0].max(dim=4)[0].tolist()==[[[[5.,7.],[13.,15.]]]]\n', globals()), end='')


<!-- dd:dd-q1267 -->

### Problem 1267 · faded — your turn

Return positive parts of locally pooled values. x: float (batch,channels,height,width); k: square kernel size; s: stride; p: zero padding on every side, 0 ≤ p ≤ k//2. The kernel fits the padded image.


In [ ]:
import torch as t

def solve(x,k,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1267)


In [ ]:
#@title 💡 Solution — Problem 1267
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,k,s,p):
    b,c,h,w=x.shape
    xx=t.full((b,c,h+2*p,w+2*p),float("-inf"),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+w]=x
    bs,cs,hs,ws=xx.stride()
    win=xx.as_strided((b,c,1+(h+2*p-k)//s,1+(w+2*p-k)//s,k,k),(bs,cs,s*hs,s*ws,hs,ws))
    y=win.max(dim=5)[0].max(dim=4)[0]
    return y.clamp(min=0)


<!-- dd:dd-q1269 -->

### Problem 1269 · independent

Return the global average features with the mean across channels removed for each example, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1269)


In [ ]:
#@title 💡 Solution — Problem 1269
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    g=x.mean(dim=(2,3))
    return g-g.mean(dim=1,keepdim=True)


<!-- dd:dd-q1270 -->

### Problem 1270 · independent

Return local MIN pooling with kernel k, stride s and padding p: the smallest value in every window, where padded positions never win. x: float (batch,channels,height,width); k: square kernel size; s: stride; p: zero padding on every side, 0 ≤ p ≤ k//2. The kernel fits the padded image.


In [ ]:
import torch as t

def solve(x,k,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1270)


In [ ]:
#@title 💡 Solution — Problem 1270
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,k,s,p):
    b,c,h,w=x.shape
    xx=t.full((b,c,h+2*p,w+2*p),float("inf"),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+w]=x
    bs,cs,hs,ws=xx.stride()
    win=xx.as_strided((b,c,1+(h+2*p-k)//s,1+(w+2*p-k)//s,k,k),(bs,cs,s*hs,s*ws,hs,ws))
    return win.min(dim=5)[0].min(dim=4)[0]


<!-- dd:dd-q1271 -->

### Problem 1271 · independent

Return the spatial range of each channel, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1271)


In [ ]:
#@title 💡 Solution — Problem 1271
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.max(dim=3)[0].max(dim=2)[0]-x.min(dim=3)[0].min(dim=2)[0]


<!-- dd:dd-q1272 -->

### Problem 1272 · independent

Return the difference between global max and global mean of each channel. x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1272)


In [ ]:
#@title 💡 Solution — Problem 1272
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.max(dim=3)[0].max(dim=2)[0]-x.mean(dim=(2,3))


<!-- dd:dd-q1273 -->

### Problem 1273 · independent

Return the fraction of positive pixels in each example/channel, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1273)


In [ ]:
#@title 💡 Solution — Problem 1273
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return (x>0).to(x.dtype).mean(dim=(2,3))


<!-- dd:dd-q1274 -->

### Problem 1274 · independent

Return global averages of locally max-pooled maps, shape (batch,channels). x: float (batch,channels,height,width); k: square kernel size; s: stride; p: zero padding on every side, 0 ≤ p ≤ k//2. The kernel fits the padded image.


In [ ]:
import torch as t

def solve(x,k,s,p):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1274)


In [ ]:
#@title 💡 Solution — Problem 1274
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,k,s,p):
    b,c,h,w=x.shape
    xx=t.full((b,c,h+2*p,w+2*p),float("-inf"),dtype=x.dtype)
    xx[:,:,p:p+h,p:p+w]=x
    bs,cs,hs,ws=xx.stride()
    win=xx.as_strided((b,c,1+(h+2*p-k)//s,1+(w+2*p-k)//s,k,k),(bs,cs,s*hs,s*ws,hs,ws))
    y=win.max(dim=5)[0].max(dim=4)[0]
    return y.mean(dim=(2,3))


<!-- dd:dd-q1275 -->

### Problem 1275 · independent

Return each channel’s spatial maximum location as a flattened row-major index, shape (batch,channels). Ties choose first. x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1275)


In [ ]:
#@title 💡 Solution — Problem 1275
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return x.reshape(x.shape[0],x.shape[1],-1).argmax(dim=2)


<!-- dd:dd-q1276 -->

### Problem 1276 · independent

Return unit-length global-average feature vectors, shape (batch,channels). A zero vector stays zero. x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1276)


In [ ]:
#@title 💡 Solution — Problem 1276
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    y=x.mean(dim=(2,3))
    n=y.norm(dim=1,keepdim=True)
    return y/t.where(n>0,n,t.ones_like(n))


<!-- dd:dd-q1277 -->

### Problem 1277 · independent

Return root-mean-square activation per example/channel, shape (batch,channels). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1277)


In [ ]:
#@title 💡 Solution — Problem 1277
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    return t.sqrt((x*x).mean(dim=(2,3)))


#### Common mistakes

- **Global pooling returns one number per image.** It returns one per channel per image, shape `(batch, channels)`.
- **Zero padding is fine for max pooling.** A zero beats every negative activation; pad with `-inf`.
- **Pooling reduces the channel axis.** It never does; channels are distinct features and stay separate.
- **`x.max(dim=(2, 3))` works like `mean`.** `max` takes one axis at a time; reduce twice and index `[0]` for the values.


<!-- dd:dd-kp-cnn-batch-normalization -->

## Batch normalization

`cnn.batch-normalization`


<!-- dd:dd-seg-cnn-batch-normalization-0 -->

### One distribution per channel


BatchNorm treats each channel as its own distribution. For a batch `x` of shape `(batch, channels, height, width)`, the statistics of channel `c` are taken over every example and every pixel — axes `0`, `2` and `3` — while channels are never mixed. The procedure is: compute the per-channel mean, subtract it, compute the per-channel variance of the centred values, divide by `sqrt(var + eps)`, then apply a learned scale `g` and shift `b` per channel. Every per-channel vector has shape `(channels,)` and must be broadcast back as `[None, :, None, None]` to line up with the channel axis.

The reason the reduction spans batch AND pixels is that a channel is one feature detector; its output at any position on any image is a sample from the same distribution. The variance is the *population* variance — the plain mean of squared deviations, with no `n-1` correction — because the batch is not a sample of some larger truth here; it is the thing being normalized. `eps` sits inside the square root so that a constant channel, whose variance is exactly zero, divides by `sqrt(eps)` rather than by zero and comes out as all zeros, which then equals the shift `b`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[1.,3.]],[[2.,6.]]]])\nmean=x.mean(dim=(0,2,3),keepdim=True)\nvar=((x-mean)**2).mean(dim=(0,2,3),keepdim=True)\nz=(x-mean)/t.sqrt(var+1e-5)\nprint(z)\n# Hidden checks\nassert t.allclose(z,t.tensor([[[[-1.,1.]],[[-1.,1.]]]]),atol=1e-3)\n', globals()), end='')


We normalize a batch of two images with one channel and two pixels each, so the channel's four values are `5, 7, 1, 3`. Their mean is `4`, and the squared deviations are `1, 9, 9, 1`, so the population variance is `5`. Predict the sign of each normalized value before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[5.,7.]]],[[[1.,3.]]]])\nmean=x.mean(dim=(0,2,3))\nvar=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))\nprint(mean, var)\n# Hidden checks\nassert mean.tolist()==[4.] and var.tolist()==[5.]\n', globals()), end='')




Dividing by `sqrt(var + eps)` leaves values roughly between `-1.4` and `1.4`; the learned scale and shift then move them wherever the next layer wants. With `g = 2` and `b = 1` the pixel that was at the mean becomes exactly `1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('g=t.tensor([2.]); b=t.tensor([1.]); eps=1e-5\nz=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)\ny=z*g[None,:,None,None]+b[None,:,None,None]\nprint(y)\n# Hidden checks\nassert t.allclose(y[0,0,0],t.tensor([1.894,3.683]),atol=1e-2)\n', globals()), end='')


<!-- dd:dd-q1297 -->

### Problem 1297 · faded — your turn

Return per-channel batch means, shape (channels,). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1297)


In [ ]:
#@title 💡 Solution — Problem 1297
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    mean=x.mean(dim=(0,2,3))
    return mean


We normalize a batch of two images with one channel and two pixels each, so the channel's four values are `5, 7, 1, 3`. Their mean is `4`, and the squared deviations are `1, 9, 9, 1`, so the population variance is `5`. Predict the sign of each normalized value before running.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[5.,7.]]],[[[1.,3.]]]])\nmean=x.mean(dim=(0,2,3))\nvar=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))\nprint(mean, var)\n# Hidden checks\nassert mean.tolist()==[4.] and var.tolist()==[5.]\n', globals()), end='')




Dividing by `sqrt(var + eps)` leaves values roughly between `-1.4` and `1.4`; the learned scale and shift then move them wherever the next layer wants. With `g = 2` and `b = 1` the pixel that was at the mean becomes exactly `1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('g=t.tensor([2.]); b=t.tensor([1.]); eps=1e-5\nz=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)\ny=z*g[None,:,None,None]+b[None,:,None,None]\nprint(y)\n# Hidden checks\nassert t.allclose(y[0,0,0],t.tensor([1.894,3.683]),atol=1e-2)\n', globals()), end='')


<!-- dd:dd-q1298 -->

### Problem 1298 · faded — your turn

Return training-mode normalized and affine-transformed x. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1298)


In [ ]:
#@title 💡 Solution — Problem 1298
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,eps):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return y


<!-- dd:dd-seg-cnn-batch-normalization-1 -->

### Running statistics remember past batches


At evaluation time there may be no batch to take statistics from — a single image, say — so BatchNorm keeps a *running* mean and variance per channel and updates them during training. The update is an exponential moving average: `running = (1 - momentum) * running + momentum * batch_stat`. With `momentum = 0.1` each new batch nudges the estimate a tenth of the way toward itself; with `momentum = 1` the estimate is simply the last batch.

The update is bookkeeping, not computation the network learns from, so it is done in place on the buffer and wrapped in `with t.no_grad():`. `with` opens a block in which a context is active and closes it when the block ends; `t.no_grad()` is the context in which autograd stops recording. The reason it is needed is that the batch mean was computed from `x`, which may carry a gradient graph; writing it into the buffer without `no_grad` would splice that graph into the buffer's history, and the next training step would try to backpropagate into the previous batch. Wrapping only the update, not the normalization, keeps the gradient path through the output intact.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nrunning=t.tensor([2.,6.])\nbatch_mean=t.tensor([4.,2.])\nmomentum=.25\nwith t.no_grad():\n    running.copy_((1-momentum)*running+momentum*batch_mean)\nprint(running)\n# Hidden checks\nassert running.tolist()==[2.5,5.]\n', globals()), end='')


We watch the running mean converge toward a repeated batch. The buffer starts at `0`, the batch mean is `8`, and `momentum = 0.5` moves halfway each step. After one update the estimate is `4`; predict the second.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nrunning=t.tensor([0.])\nbatch_mean=t.tensor([8.])\nmomentum=.5\nwith t.no_grad():\n    running.copy_((1-momentum)*running+momentum*batch_mean)\nprint(running)\n# Hidden checks\nassert running.tolist()==[4.]\n', globals()), end='')




The second step starts from the updated value, not the original: `(1-0.5)*4 + 0.5*8 = 6`. Each application closes half the remaining gap, which is why the estimate never overshoots and never quite arrives.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('with t.no_grad():\n    running.copy_((1-momentum)*running+momentum*batch_mean)\nprint(running)\n# Hidden checks\nassert running.tolist()==[6.]\n', globals()), end='')


<!-- dd:dd-q1299 -->

### Problem 1299 · faded — your turn

Return the updated running mean after this batch. x: float (batch,channels,height,width); rm: running mean (channels,); momentum in (0,1]. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,rm,momentum):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1299)


In [ ]:
#@title 💡 Solution — Problem 1299
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,rm,momentum):
    return (1-momentum)*rm+momentum*x.mean(dim=(0,2,3))


<!-- dd:dd-seg-cnn-batch-normalization-2 -->

### Training uses the batch; evaluation uses the memory


A module carries a flag, `self.training`, that `m.train()` sets to `True` and `m.eval()` sets to `False`. BatchNorm's forward branches on it: in training mode it normalizes with the current batch's statistics and updates the running buffers; in evaluation mode it normalizes with the running buffers and changes nothing. The learned scale and shift are applied identically in both modes.

The reason for two behaviours is what each mode is for. Training wants the actual distribution of the batch, so the network learns against what it will see. Evaluation wants each prediction to depend only on its own input: if the batch statistics were used, the output for one image would change depending on which other images happened to share its batch, and a model served one request at a time would divide by a variance of zero. So evaluation uses the statistics remembered from training, and the buffers are frozen — which is also why evaluation must not update them.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nm=t.nn.Module()\nm.eval()\nif m.training:\n    label="batch statistics"\nelse:\n    label="running statistics"\nprint(label)\n# Hidden checks\nassert label=="running statistics"\n', globals()), end='')


We normalize the same one-channel batch in both modes and compare. In evaluation mode the stored mean `1` and variance `4` are used, so a pixel at `1` maps to zero and a pixel at `3` maps to `2 / sqrt(4 + eps)`, just under `1`.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('import torch as t\nx=t.tensor([[[[1.,3.]]]])\nrm=t.tensor([1.]); rv=t.tensor([4.]); eps=1e-5\nz_eval=(x-rm[None,:,None,None])/t.sqrt(rv[None,:,None,None]+eps)\nprint(z_eval)\n# Hidden checks\nassert t.allclose(z_eval,t.tensor([[[[0.,1.]]]]),atol=1e-3)\n', globals()), end='')




In training mode the batch's own statistics apply — mean `2`, variance `1` — so the same two pixels land symmetrically at `-1` and `1`. Same input, different output: that is the flag doing its job.


In [ ]:
# Validation wrapper; the authored example is in cell metadata.
_dd_check_scope = {"__name__": "_dd_lesson_checks"}
exec('"""Execute authored lesson cells with the same output/check contract as the UI."""\nimport ast\nimport contextlib\nimport io\nimport re\n\nMARKER = \'\\n# Hidden checks\\n\'\n\n\ndef split_checks(source):\n    visible, marker, hidden = source.partition(MARKER)\n    return visible, hidden if marker else \'\'\n\n\ndef _assertions(tree):\n    for node in ast.walk(tree):\n        if isinstance(node, ast.Assert):\n            if isinstance(node.test, ast.Constant):\n                raise AssertionError(\'An assertion must check a result, not a constant\')\n            if isinstance(node.test, ast.Compare) and len(node.test.comparators) == 1:\n                if ast.dump(node.test.left) == ast.dump(node.test.comparators[0]):\n                    raise AssertionError(\'An assertion must not compare a value with itself\')\n            yield node\n\n\ndef run_checked(source, namespace=None, *, require_assert=True):\n    """Echo the last expression, capture actual output, then run hidden checks.\n\n    Hidden checks get a shallow namespace copy so helper names cannot overwrite\n    the next example\'s names. Their assertions still inspect the actual values.\n    Count assertions reached at runtime: an uncalled function or dead branch\n    containing `assert` is not a check of this cell.\n    """\n    if not __debug__:\n        raise AssertionError(\'Lesson checks require assertions enabled\')\n    ns = namespace if namespace is not None else {}\n    visible, hidden = split_checks(source)\n    trees = [ast.parse(visible, \'<lesson>\'), ast.parse(hidden, \'<lesson checks>\')]\n    sites = [(tree, list(_assertions(tree))) for tree in trees]\n    if require_assert and not any(nodes for _, nodes in sites):\n        raise AssertionError(\'Runnable lesson cell has no assertion\')\n    reached = []\n\n    class CountAssertions(ast.NodeTransformer):\n        def visit_Assert(self, node):\n            call = ast.Expr(ast.Call(ast.Name(\'_delta_reached\', ast.Load()), [], []))\n            return [ast.copy_location(call, node), node]\n\n    trees = [ast.fix_missing_locations(CountAssertions().visit(tree)) for tree in trees]\n    ns[\'_delta_reached\'] = lambda: reached.append(True)\n    output = io.StringIO()\n    with contextlib.redirect_stdout(output):\n        tree = trees[0]\n        if tree.body and isinstance(tree.body[-1], ast.Expr):\n            head = ast.Module(body=tree.body[:-1], type_ignores=[])\n            exec(compile(head, \'<lesson>\', \'exec\'), ns)\n            value = eval(compile(ast.Expression(tree.body[-1].value), \'<lesson>\', \'eval\'), ns)\n            if value is not None:\n                print(repr(value))\n        else:\n            exec(compile(tree, \'<lesson>\', \'exec\'), ns)\n    check_ns = dict(ns, _delta_output=output.getvalue())\n    exec(compile(trees[1], \'<lesson checks>\', \'exec\'), check_ns)\n    if require_assert and not reached:\n        raise AssertionError(\'No assertion executed; a dormant assert does not check this cell\')\n    return output.getvalue()\n\n\ndef check_compiled_examples(directory):\n    """All runnable teaching cells, fresh namespace for each concept page."""\n    import json\n    from pathlib import Path\n    directory = Path(directory)\n    data = json.loads((directory / \'lessons_structured.json\').read_text())\n    authored = {}\n    for path in directory.glob(\'*/kp-*.md\'):\n        text = path.read_text()\n        kc = re.search(r\'^kc: (.+)$\', text, re.M)\n        if kc:\n            authored[kc[1]] = re.findall(r\'^```python\\s*\\n(.*?)^```\', text, re.M | re.S)\n    count = 0\n    for lesson in data[\'lessons\']:\n        for kp in lesson[\'kps\']:\n            compiled = [source for segment in kp[\'segments\']\n                        for field in (\'concept_markdown\', \'worked_example_markdown\')\n                        for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S)]\n            assert authored.get(kp[\'kc\']) == compiled, (\n                f"{kp[\'kc\']}: teaching source changed; validate and recompile lessons")\n            for segment in kp[\'segments\']:\n                ns = {}\n                for field in (\'concept_markdown\', \'worked_example_markdown\'):\n                    for source in re.findall(r\'^```python\\s*\\n(.*?)^```\', segment[field], re.M | re.S):\n                        try:\n                            run_checked(source, ns)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/{segment[\'concept_id\']}/{field}: {exc}") from exc\n                        count += 1\n            for field in (\'guided_items\', \'applied_items\', \'solo_items\'):\n                for item in kp.get(field, []):\n                    source = item.get(\'worked_example_code\')\n                    if source:\n                        try:\n                            run_checked(source)\n                        except Exception as exc:\n                            raise AssertionError(f"{kp[\'kc\']}/q{item[\'question_id\']}: {exc}") from exc\n                        count += 1\n    return count\n\n\nif __name__ == \'__main__\':\n    from pathlib import Path\n    print(f\'PASS: {check_compiled_examples(Path(__file__).parent)} runnable cells asserted\')\n', _dd_check_scope)
print(_dd_check_scope['run_checked']('mean=x.mean(dim=(0,2,3)); var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))\nz_train=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)\nprint(z_train)\n# Hidden checks\nassert t.allclose(z_train,t.tensor([[[[-1.,1.]]]]),atol=1e-3)\n', globals()), end='')


<!-- dd:dd-q1300 -->

### Problem 1300 · faded — your turn

Return evaluation-mode normalized and affine-transformed x. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); rm: running mean (channels,); rv: running variance (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,rm,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1300)


In [ ]:
#@title 💡 Solution — Problem 1300
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,rm,rv,eps):
    mean=rm
    var=rv
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return y


<!-- dd:dd-q1301 -->

### Problem 1301 · independent

Return the training-mode BatchNorm output followed by ReLU, preserving the shape of x. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1301)


In [ ]:
#@title 💡 Solution — Problem 1301
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,eps):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return y.clamp(min=0)


<!-- dd:dd-q1302 -->

### Problem 1302 · independent

Return the evaluation-mode BatchNorm output averaged over both spatial axes, shape (batch,channels). x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); rm: running mean (channels,); rv: running variance (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,rm,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1302)


In [ ]:
#@title 💡 Solution — Problem 1302
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,rm,rv,eps):
    z=(x-rm[None,:,None,None])/t.sqrt(rv[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return y.mean(dim=(2,3))


<!-- dd:dd-q1303 -->

### Problem 1303 · independent

Return population variance of each input channel, shape (channels,). x: float (batch,channels,height,width).


In [ ]:
import torch as t

def solve(x):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1303)


In [ ]:
#@title 💡 Solution — Problem 1303
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    return var


<!-- dd:dd-q1304 -->

### Problem 1304 · independent

Return the updated running variance after this batch. x: float (batch,channels,height,width); rv: running variance (channels,); momentum in (0,1]. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,rv,momentum):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1304)


In [ ]:
#@title 💡 Solution — Problem 1304
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,rv,momentum):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    return (1-momentum)*rv+momentum*var


<!-- dd:dd-q1305 -->

### Problem 1305 · independent

Return training output minus evaluation output for the same input. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); rm: running mean (channels,); rv: running variance (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,rm,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1305)


In [ ]:
#@title 💡 Solution — Problem 1305
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,rm,rv,eps):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    training=y
    mean=rm
    var=rv
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    return training-y


<!-- dd:dd-q1306 -->

### Problem 1306 · independent

Return the running mean after observing this same batch twice. Each update uses the result of the previous update. x: float (batch,channels,height,width); rm: running mean (channels,); momentum in (0,1]. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,rm,momentum):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1306)


In [ ]:
#@title 💡 Solution — Problem 1306
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,rm,momentum):
    mean=x.mean(dim=(0,2,3))
    first=(1-momentum)*rm+momentum*mean
    return (1-momentum)*first+momentum*mean


<!-- dd:dd-q1307 -->

### Problem 1307 · independent

Return the effective scale per channel for evaluation: the multiplier that converts each input value into output before adding a constant offset. g: scale (channels,); rv: running variance (channels,); eps > 0.


In [ ]:
import torch as t

def solve(g,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1307)


In [ ]:
#@title 💡 Solution — Problem 1307
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(g,rv,eps):
    return g/t.sqrt(rv+eps)


<!-- dd:dd-q1308 -->

### Problem 1308 · independent

Return the effective offset per channel for evaluation, so output equals input times effective scale plus this offset. g: scale (channels,); b: shift (channels,); rm: running mean (channels,); rv: running variance (channels,); eps > 0.


In [ ]:
import torch as t

def solve(g,b,rm,rv,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1308)


In [ ]:
#@title 💡 Solution — Problem 1308
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(g,b,rm,rv,eps):
    return b-rm*g/t.sqrt(rv+eps)


<!-- dd:dd-q1309 -->

### Problem 1309 · independent

Return training-mode channel means and population variances after the affine transform, shape (2,channels), means first. x: float (batch,channels,height,width); g: scale (channels,); b: shift (channels,); eps > 0. Variance is the population variance.


In [ ]:
import torch as t

def solve(x,g,b,eps):
    pass


In [ ]:
# Did it work? Run this. (NameError → run the checker cell at
# the top of the notebook first: Runtime ▸ Run before.)
dd_check(1309)


In [ ]:
#@title 💡 Solution — Problem 1309
# Running this rebinds `solve` to the reference answer. Re-run your
# own cell before dd_check() again, or you are checking this one.
import torch as t

def solve(x,g,b,eps):
    mean=x.mean(dim=(0,2,3))
    var=((x-mean[None,:,None,None])**2).mean(dim=(0,2,3))
    z=(x-mean[None,:,None,None])/t.sqrt(var[None,:,None,None]+eps)
    y=z*g[None,:,None,None]+b[None,:,None,None]
    ym=y.mean(dim=(0,2,3))
    yv=((y-ym[None,:,None,None])**2).mean(dim=(0,2,3))
    return t.stack((ym,yv))


#### Common mistakes

- **Statistics are per image.** They are per channel, pooled over every example in the batch and every pixel; a per-image mean would erase the difference between a bright image and a dark one.
- **Variance uses `n-1`.** BatchNorm uses the population variance; `x.var()`'s default correction gives a different number.
- **Evaluation mode still updates the buffers.** It reads them and leaves them alone; only training mode writes.
- **The running update is part of the gradient path.** It is bookkeeping done under `no_grad`; the normalized output, not the buffer write, is what gradients flow through.
